In [98]:
from itertools import combinations

def build_pair_features(candidate_df):
    rows = []

    for (document_id, label_id), group in candidate_df.groupby(
        ["document_id", "label_id"]
    ):

        group = group.reset_index(drop=True)

        # At most 5 retrieved clauses → max 10 pairs
        for i, j in combinations(range(len(group)), 2):

            a = group.iloc[i]
            b = group.iloc[j]

            # Skip identical clauses
            if a["span_id"] == b["span_id"]:
                continue

            # Modality conflict
            mod_conflict = modality_conflict(
                a["modality"],
                b["modality"]
            )

            # Explicit negation conflict
            neg_conflict = explicit_negation_conflict(
                a["clause"],
                b["clause"]
            )

            # Proposition similarity
            prop_a = normalize_clause(a["proposition"])
            prop_b = normalize_clause(b["proposition"])

            if prop_a == prop_b and prop_a:
                prop_sim = 1.0
            else:
                # Simple token Jaccard similarity
                tokens_a = set(re.findall(r"\b[a-zA-Z]{3,}\b", prop_a))
                tokens_b = set(re.findall(r"\b[a-zA-Z]{3,}\b", prop_b))

                if tokens_a and tokens_b:
                    prop_sim = len(tokens_a & tokens_b) / len(
                        tokens_a | tokens_b
                    )
                else:
                    prop_sim = 0.0

            # Condition similarity
            cond_a = set(a["conditions"])
            cond_b = set(b["conditions"])

            if not cond_a and not cond_b:
                cond_sim = 1.0
            elif not cond_a or not cond_b:
                cond_sim = 0.0
            else:
                cond_sim = len(cond_a & cond_b) / len(
                    cond_a | cond_b
                )

            # Party similarity
            parties_a = set(a["parties"])
            parties_b = set(b["parties"])

            if not parties_a and not parties_b:
                party_sim = 1.0
            elif not parties_a or not parties_b:
                party_sim = 0.0
            else:
                party_sim = len(parties_a & parties_b) / len(
                    parties_a | parties_b
                )

            rows.append({
                "document_id": document_id,
                "label_id": label_id,
                "target": int(a["target"]),
                "span_a": int(a["span_id"]),
                "span_b": int(b["span_id"]),
                "text_a": a["clause"],
                "text_b": b["clause"],
                "retrieval_a": float(a["retrieval_score"]),
                "retrieval_b": float(b["retrieval_score"]),
                "modality_a": a["modality"],
                "modality_b": b["modality"],
                "modality_conflict": mod_conflict,
                "negation_conflict": neg_conflict,
                "proposition_similarity": prop_sim,
                "condition_similarity": cond_sim,
                "party_similarity": party_sim
            })

    return pd.DataFrame(rows)

In [99]:
dev_pair_df = build_pair_features(
    dev_structured_df
)

print("Dev clause pairs:", len(dev_pair_df))

print(
    "Unique hypothesis examples:",
    dev_pair_df[
        ["document_id", "label_id"]
    ].drop_duplicates().shape[0]
)

display(
    dev_pair_df.head(10)
)

Dev clause pairs: 10370
Unique hypothesis examples: 1037


,document_id,label_id,target,span_a,span_b,text_a,text_b,retrieval_a,retrieval_b,modality_a,modality_b,modality_conflict,negation_conflict,proposition_similarity,condition_similarity,party_similarity
0,10,nda-1,2,44,90,"Upon the Disclosing Party’s written request, t...","Additionally, nothing in this CNDA constitutes...",0.178320,0.168049,obligation,none,0,0,0.100000,1.0,0.666667
1,10,nda-1,2,44,30,"Upon the Disclosing Party’s written request, t...",(d) information that the Disclosing Party auth...,0.178320,0.149521,obligation,none,0,0,0.111111,1.0,1.000000
2,10,nda-1,2,44,49,"Upon the Disclosing Party’s written request, t...",The Receiving Party acknowledges and agrees th...,0.178320,0.137766,obligation,obligation,0,0,0.100000,1.0,1.000000
3,10,nda-1,2,44,22,"Upon the Disclosing Party’s written request, t...",ARTICLE 2- CONFIDENTIAL INFORMATION,0.178320,0.126756,obligation,none,0,0,0.058824,1.0,0.000000
4,10,nda-1,2,90,30,"Additionally, nothing in this CNDA constitutes...",(d) information that the Disclosing Party auth...,0.168049,0.149521,none,none,0,0,0.200000,1.0,0.666667
5,10,nda-1,2,90,49,"Additionally, nothing in this CNDA constitutes...",The Receiving Party acknowledges and agrees th...,0.168049,0.137766,none,obligation,0,0,0.100000,1.0,0.666667
6,10,nda-1,2,90,22,"Additionally, nothing in this CNDA constitutes...",ARTICLE 2- CONFIDENTIAL INFORMATION,0.168049,0.126756,none,none,0,0,0.166667,1.0,0.000000
7,10,nda-1,2,30,49,(d) information that the Disclosing Party auth...,The Receiving Party acknowledges and agrees th...,0.149521,0.137766,none,obligation,0,0,0.111111,1.0,1.000000
8,10,nda-1,2,30,22,(d) information that the Disclosing Party auth...,ARTICLE 2- CONFIDENTIAL INFORMATION,0.149521,0.126756,none,none,0,0,0.111111,1.0,0.000000
9,10,nda-1,2,49,22,The Receiving Party acknowledges and agrees th...,ARTICLE 2- CONFIDENTIAL INFORMATION,0.137766,0.126756,obligation,none,0,0,0.000000,1.0,0.000000


In [100]:
dev_pair_df["structured_conflict_score"] = (
    0.35 * dev_pair_df["proposition_similarity"]
    +
    0.25 * dev_pair_df["modality_conflict"]
    +
    0.20 * dev_pair_df["negation_conflict"]
    +
    0.10 * dev_pair_df["condition_similarity"]
    +
    0.10 * dev_pair_df["party_similarity"]
)

display(
    dev_pair_df.sort_values(
        "structured_conflict_score",
        ascending=False
    )[
        [
            "document_id",
            "span_a",
            "span_b",
            "structured_conflict_score",
            "proposition_similarity",
            "modality_conflict",
            "negation_conflict",
            "condition_similarity",
            "party_similarity",
            "text_a",
            "text_b"
        ]
    ].head(20)
)

,document_id,span_a,span_b,structured_conflict_score,proposition_similarity,modality_conflict,negation_conflict,condition_similarity,party_similarity,text_a,text_b
7240,590,7,11,0.825000,0.500000,1,1,1.0,1.0,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7260,590,11,7,0.825000,0.500000,1,1,1.0,1.0,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7147,590,11,7,0.825000,0.500000,1,1,1.0,1.0,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
6810,582,13,21,0.807746,0.450704,1,1,1.0,1.0,"You and the Company also each agree, on behalf...",4. In the event you or any of your Representat...
361,13,43,111,0.766667,0.333333,1,1,1.0,1.0,A Disclosing Party may disclose or discuss a P...,Disclosure of Confidential Information of any ...
4689,440,19,7,0.766667,0.333333,1,1,1.0,1.0,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
4647,440,19,7,0.766667,0.333333,1,1,1.0,1.0,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
9133,71,9,10,0.756061,0.303030,1,1,1.0,1.0,Prospect and Prospect’s Broker acknowledge tha...,Prospect’s and Prospect’s Broker’s disseminati...
9160,71,10,9,0.756061,0.303030,1,1,1.0,1.0,Prospect’s and Prospect’s Broker’s disseminati...,Prospect and Prospect’s Broker acknowledge tha...
9113,71,9,10,0.756061,0.303030,1,1,1.0,1.0,Prospect and Prospect’s Broker acknowledge tha...,Prospect’s and Prospect’s Broker’s disseminati...


In [101]:
dev_example_scores = (
    dev_pair_df
    .groupby(
        ["document_id", "label_id"],
        as_index=False
    )["structured_conflict_score"]
    .max()
    .rename(
        columns={
            "structured_conflict_score":
            "max_conflict_score"
        }
    )
)

# Recover the true target
dev_targets = (
    dev_df[
        [
            "document_id",
            "label_id",
            "target"
        ]
    ]
    .drop_duplicates()
)

dev_example_scores = dev_example_scores.merge(
    dev_targets,
    on=["document_id", "label_id"],
    how="left"
)

print(
    "Evaluation examples:",
    len(dev_example_scores)
)

display(dev_example_scores.head())

Evaluation examples: 1037


,document_id,label_id,max_conflict_score,target
0,10,nda-1,0.238889,2
1,10,nda-10,0.716667,1
2,10,nda-11,0.658333,0
3,10,nda-12,0.433333,0
4,10,nda-13,0.433333,1


In [102]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

threshold_results = []

for threshold in np.arange(
    0.20,
    0.81,
    0.05
):

    pred_conflict = (
        dev_example_scores["max_conflict_score"]
        >= threshold
    ).astype(int)

    true_conflict = (
        dev_example_scores["target"] == 2
    ).astype(int)

    precision = precision_score(
        true_conflict,
        pred_conflict,
        zero_division=0
    )

    recall = recall_score(
        true_conflict,
        pred_conflict,
        zero_division=0
    )

    f1 = f1_score(
        true_conflict,
        pred_conflict,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(
    threshold_results
)

display(
    threshold_df.sort_values(
        "f1",
        ascending=False
    )
)

,threshold,precision,recall,f1
0,0.20,0.092683,1.000000,0.169643
1,0.25,0.090807,0.852632,0.164134
3,0.35,0.081081,0.536842,0.140884
2,0.30,0.078148,0.568421,0.137405
4,0.40,0.074010,0.452632,0.127219
8,0.60,0.080997,0.273684,0.125000
5,0.45,0.072464,0.421053,0.123648
7,0.55,0.073375,0.368421,0.122378
9,0.65,0.087963,0.200000,0.122186
6,0.50,0.072125,0.389474,0.121711


In [103]:
best_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

BEST_THRESHOLD = float(
    best_row["threshold"]
)

print("Best Dev threshold:", BEST_THRESHOLD)
print("Precision:", best_row["precision"])
print("Recall:", best_row["recall"])
print("F1:", best_row["f1"])

Best Dev threshold: 0.2
Precision: 0.09268292682926829
Recall: 1.0
F1: 0.16964285714285715


In [106]:
def proposition_token_set(text):
    text = normalize_clause(text)

    stop_words = {
        "the", "and", "or", "of", "to", "a", "an", "any",
        "such", "this", "that", "shall", "must", "may", "will",
        "not", "be", "is", "are", "for", "in", "on", "with",
        "by", "from", "under", "as", "its", "their",
        "party", "parties"
    }

    tokens = re.findall(r"\b[a-z]{3,}\b", text)

    return {
        token for token in tokens
        if token not in stop_words
    }


def proposition_overlap(text_a, text_b):
    a = proposition_token_set(text_a)
    b = proposition_token_set(text_b)

    if not a or not b:
        return 0.0

    return len(a & b) / min(len(a), len(b))

In [108]:
dev_pair_df["proposition_overlap"] = dev_pair_df.apply(
    lambda r: proposition_overlap(
        r["text_a"],
        r["text_b"]
    ),
    axis=1
)

print("Proposition overlap:")
print(dev_pair_df["proposition_overlap"].describe())

display(
    dev_pair_df.sort_values(
        "proposition_overlap",
        ascending=False
    )[
        [
            "document_id",
            "span_a",
            "span_b",
            "proposition_similarity",
            "proposition_overlap",
            "modality_a",
            "modality_b",
            "text_a",
            "text_b"
        ]
    ].head(15)
)

Proposition overlap:
count    10370.000000
mean         0.306588
std          0.262480
min          0.000000
25%          0.111111
50%          0.250000
75%          0.439375
max          1.000000
Name: proposition_overlap, dtype: float64


,document_id,span_a,span_b,proposition_similarity,proposition_overlap,modality_a,modality_b,text_a,text_b
76,10,42,23,0.054054,1.0,prohibition,none,The Receiving Party will not disclose to any P...,2.01 Confidential Information
10323,9,15,76,0.166667,1.0,none,obligation,Confidential Information.,"Upon request of BPS at any time, VENDOR shall ..."
73,10,22,23,0.666667,1.0,none,none,ARTICLE 2- CONFIDENTIAL INFORMATION,2.01 Confidential Information
10301,9,15,50,0.181818,1.0,none,none,Confidential Information.,d) Requiring that users be uniquely identified...
3535,39,12,26,0.030303,1.0,none,prohibition,1. Purpose,A. “Confidential Information” as outlined here...
53,10,36,25,0.214286,1.0,prohibition,none,"The Receiving Party shall not, in any manner, ...",2.02 Not Confidential Information
43,10,27,23,0.090909,1.0,none,none,(a) information relating to the Disclosing Par...,2.01 Confidential Information
10330,9,15,61,0.125000,1.0,none,prohibition,Confidential Information.,VENDOR shall protect the Confidential Informat...
30,10,22,23,0.666667,1.0,none,none,ARTICLE 2- CONFIDENTIAL INFORMATION,2.01 Confidential Information
59,10,22,25,0.500000,1.0,none,none,ARTICLE 2- CONFIDENTIAL INFORMATION,2.02 Not Confidential Information


In [109]:
dev_pair_df["strict_conflict_score"] = (
    0.55 * dev_pair_df["proposition_overlap"]
    + 0.20 * dev_pair_df["proposition_similarity"]
    + 0.15 * dev_pair_df["modality_conflict"]
    + 0.10 * dev_pair_df["condition_similarity"]
)

# Hard suppression:
# Opposite modality alone should NOT create a strong conflict.

dev_pair_df.loc[
    dev_pair_df["proposition_overlap"] < 0.35,
    "strict_conflict_score"
] *= 0.25

print("Strict conflict score:")
print(dev_pair_df["strict_conflict_score"].describe())

Strict conflict score:
count    10370.000000
mean         0.187179
std          0.211089
min          0.000000
25%          0.036075
50%          0.065625
75%          0.363649
max          0.900000
Name: strict_conflict_score, dtype: float64


In [111]:
dev_strict_scores = (
    dev_pair_df
    .groupby(
        ["document_id", "label_id"],
        as_index=False
    )["strict_conflict_score"]
    .max()
    .rename(
        columns={
            "strict_conflict_score": "max_conflict_score"
        }
    )
)

dev_strict_scores = dev_strict_scores.merge(
    dev_targets,
    on=["document_id", "label_id"],
    how="left"
)

print("Rows:", len(dev_strict_scores))
print("\nTarget distribution:")
print(dev_strict_scores["target"].value_counts().sort_index())

display(dev_strict_scores.head())

Rows: 1037

Target distribution:
target
0    423
1    519
2     95
Name: count, dtype: int64


,document_id,label_id,max_conflict_score,target
0,10,nda-1,0.500000,2
1,10,nda-10,0.641667,1
2,10,nda-11,0.471579,0
3,10,nda-12,0.783333,0
4,10,nda-13,0.783333,1


In [112]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):

    y_true = (
        dev_strict_scores["target"] == 2
    ).astype(int)

    y_pred = (
        dev_strict_scores["max_conflict_score"] >= threshold
    ).astype(int)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.sort_values(
        "f1",
        ascending=False
    )
)

,threshold,precision,recall,f1
0,0.10,0.098305,0.915789,0.177551
3,0.25,0.098476,0.884211,0.177215
4,0.30,0.098558,0.863158,0.176915
2,0.20,0.097926,0.894737,0.176532
1,0.15,0.097926,0.894737,0.176532
8,0.50,0.103211,0.473684,0.169492
7,0.45,0.098113,0.547368,0.166400
5,0.35,0.093144,0.757895,0.165899
6,0.40,0.093272,0.642105,0.162884
9,0.55,0.098870,0.368421,0.155902


In [113]:
best_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

print("Best threshold:", best_row["threshold"])
print("Precision:", round(best_row["precision"], 4))
print("Recall:", round(best_row["recall"], 4))
print("F1:", round(best_row["f1"], 4))

Best threshold: 0.1
Precision: 0.0983
Recall: 0.9158
F1: 0.1776


In [114]:
best_threshold = best_row["threshold"]

high_conflict = dev_pair_df[
    dev_pair_df["strict_conflict_score"] >= best_threshold
].copy()

high_conflict = high_conflict.sort_values(
    "strict_conflict_score",
    ascending=False
)

print("High-confidence pair count:", len(high_conflict))

display(
    high_conflict[
        [
            "document_id",
            "span_a",
            "span_b",
            "strict_conflict_score",
            "proposition_overlap",
            "proposition_similarity",
            "modality_conflict",
            "condition_similarity",
            "modality_a",
            "modality_b",
            "text_a",
            "text_b"
        ]
    ].head(20)
)

High-confidence pair count: 3716


,document_id,span_a,span_b,strict_conflict_score,proposition_overlap,proposition_similarity,modality_conflict,condition_similarity,modality_a,modality_b,text_a,text_b
7260,590,11,7,0.900000,1.0,0.500000,1,1.0,prohibition,obligation,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7240,590,7,11,0.900000,1.0,0.500000,1,1.0,obligation,prohibition,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7147,590,11,7,0.900000,1.0,0.500000,1,1.0,prohibition,obligation,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
4689,440,19,7,0.866667,1.0,0.333333,1,1.0,prohibition,obligation,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
4647,440,19,7,0.866667,1.0,0.333333,1,1.0,prohibition,obligation,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
9695,79,10,6,0.857143,1.0,0.285714,1,1.0,prohibition,obligation,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
9814,79,10,6,0.857143,1.0,0.285714,1,1.0,prohibition,obligation,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
9790,79,10,6,0.857143,1.0,0.285714,1,1.0,prohibition,obligation,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
9400,73,133,37,0.850000,1.0,1.000000,0,1.0,none,none,“RECEIVING PARTY”,The Receiving Party:
9490,73,133,37,0.850000,1.0,1.000000,0,1.0,none,none,“RECEIVING PARTY”,The Receiving Party:


In [115]:
display(
    high_conflict[
        [
            "document_id",
            "span_a",
            "span_b",
            "strict_conflict_score",
            "proposition_overlap",
            "proposition_similarity",
            "modality_conflict",
            "condition_similarity",
            "text_a",
            "text_b"
        ]
    ].head(30)
)

,document_id,span_a,span_b,strict_conflict_score,proposition_overlap,proposition_similarity,modality_conflict,condition_similarity,text_a,text_b
7260,590,11,7,0.900000,1.0,0.500000,1,1.0,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7240,590,7,11,0.900000,1.0,0.500000,1,1.0,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7147,590,11,7,0.900000,1.0,0.500000,1,1.0,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
4689,440,19,7,0.866667,1.0,0.333333,1,1.0,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
4647,440,19,7,0.866667,1.0,0.333333,1,1.0,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
9695,79,10,6,0.857143,1.0,0.285714,1,1.0,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
9814,79,10,6,0.857143,1.0,0.285714,1,1.0,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
9790,79,10,6,0.857143,1.0,0.285714,1,1.0,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
9400,73,133,37,0.850000,1.0,1.000000,0,1.0,“RECEIVING PARTY”,The Receiving Party:
9490,73,133,37,0.850000,1.0,1.000000,0,1.0,“RECEIVING PARTY”,The Receiving Party:


In [116]:
def proposition_jaccard(text_a, text_b):
    a = proposition_token_set(text_a)
    b = proposition_token_set(text_b)

    if not a or not b:
        return 0.0

    return len(a & b) / len(a | b)


dev_pair_df["proposition_jaccard"] = dev_pair_df.apply(
    lambda r: proposition_jaccard(
        r["text_a"],
        r["text_b"]
    ),
    axis=1
)

print(
    dev_pair_df["proposition_jaccard"].describe()
)

count    10370.000000
mean         0.106337
std          0.115252
min          0.000000
25%          0.037037
50%          0.083333
75%          0.142857
max          1.000000
Name: proposition_jaccard, dtype: float64


In [117]:
dev_pair_df["aligned_conflict"] = (
    (dev_pair_df["proposition_jaccard"] >= 0.30) &
    (dev_pair_df["modality_conflict"] == 1)
).astype(int)

print(
    "Aligned conflict candidates:",
    dev_pair_df["aligned_conflict"].sum()
)

Aligned conflict candidates: 28


In [118]:
dev_pair_df["aligned_conflict_score"] = (
    0.45 * dev_pair_df["proposition_jaccard"]
    + 0.25 * dev_pair_df["proposition_similarity"]
    + 0.20 * dev_pair_df["condition_similarity"]
    + 0.10 * dev_pair_df["party_similarity"]
)

# If modality is not actually conflicting,
# suppress the score heavily.

dev_pair_df.loc[
    dev_pair_df["modality_conflict"] == 0,
    "aligned_conflict_score"
] *= 0.10

# If propositions are poorly aligned,
# suppress even further.

dev_pair_df.loc[
    dev_pair_df["proposition_jaccard"] < 0.30,
    "aligned_conflict_score"
] *= 0.10

print(
    dev_pair_df["aligned_conflict_score"].describe()
)

count    10370.000000
mean         0.008844
std          0.027857
min          0.000000
25%          0.002000
50%          0.003000
75%          0.004120
max          0.762500
Name: aligned_conflict_score, dtype: float64


In [119]:
dev_aligned_scores = (
    dev_pair_df
    .groupby(
        ["document_id", "label_id"],
        as_index=False
    )["aligned_conflict_score"]
    .max()
    .rename(
        columns={
            "aligned_conflict_score":
            "max_aligned_conflict_score"
        }
    )
)

dev_aligned_scores = dev_aligned_scores.merge(
    dev_targets,
    on=["document_id", "label_id"],
    how="left"
)

print("Rows:", len(dev_aligned_scores))

Rows: 1037


In [120]:
aligned_threshold_results = []

y_true = (
    dev_aligned_scores["target"] == 2
).astype(int)

for threshold in np.arange(0.05, 0.91, 0.05):

    y_pred = (
        dev_aligned_scores["max_aligned_conflict_score"]
        >= threshold
    ).astype(int)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    aligned_threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

aligned_threshold_df = pd.DataFrame(
    aligned_threshold_results
)

display(
    aligned_threshold_df.sort_values(
        "f1",
        ascending=False
    )
)

,threshold,precision,recall,f1
0,0.05,0.112821,0.231579,0.151724
12,0.65,0.333333,0.010526,0.020408
13,0.70,0.333333,0.010526,0.020408
14,0.75,0.333333,0.010526,0.020408
11,0.60,0.250000,0.010526,0.020202
10,0.55,0.166667,0.010526,0.019802
9,0.50,0.100000,0.010526,0.019048
8,0.45,0.083333,0.010526,0.018692
7,0.40,0.076923,0.010526,0.018519
6,0.35,0.076923,0.010526,0.018519


In [121]:
best_aligned = aligned_threshold_df.loc[
    aligned_threshold_df["f1"].idxmax()
]

print("Best threshold:", best_aligned["threshold"])
print("Precision:", round(best_aligned["precision"], 4))
print("Recall:", round(best_aligned["recall"], 4))
print("F1:", round(best_aligned["f1"], 4))

Best threshold: 0.05
Precision: 0.1128
Recall: 0.2316
F1: 0.1517


In [122]:
# Cell 78 — Inspect structured representations

sample_pairs = dev_pair_df[
    dev_pair_df["modality_conflict"] == 1
].sort_values(
    "proposition_similarity",
    ascending=False
).head(20)

display(
    sample_pairs[
        [
            "document_id",
            "span_a",
            "span_b",
            "modality_a",
            "modality_b",
            "proposition_similarity",
            "proposition_overlap",
            "proposition_jaccard",
            "condition_similarity",
            "party_similarity",
            "text_a",
            "text_b"
        ]
    ]
)

,document_id,span_a,span_b,modality_a,modality_b,proposition_similarity,proposition_overlap,proposition_jaccard,condition_similarity,party_similarity,text_a,text_b
7260,590,11,7,prohibition,obligation,0.500000,1.000000,0.750000,1.0,1.0,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7240,590,7,11,obligation,prohibition,0.500000,1.000000,0.750000,1.0,1.0,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7147,590,11,7,prohibition,obligation,0.500000,1.000000,0.750000,1.0,1.0,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
8438,64,47,59,obligation,prohibition,0.454545,0.777778,0.411765,0.0,0.0,The Information shall be kept confidential in ...,The Retained Information shall be held by the ...
8333,64,59,47,prohibition,obligation,0.454545,0.777778,0.411765,0.0,0.0,The Retained Information shall be held by the ...,The Information shall be kept confidential in ...
6810,582,13,21,prohibition,permission,0.450704,0.658537,0.428571,1.0,1.0,"You and the Company also each agree, on behalf...",4. In the event you or any of your Representat...
3870,405,20,24,prohibition,obligation,0.440000,0.692308,0.428571,0.0,1.0,You and your representatives will not use any ...,Without limiting the generality of the foregoi...
8439,64,61,59,obligation,prohibition,0.391304,0.666667,0.333333,0.0,0.0,Any and all oral Information shall continue to...,The Retained Information shall be held by the ...
1319,17,30,29,permission,prohibition,0.380952,0.600000,0.333333,0.0,1.0,h. The contractor may disclose VA sensitive in...,The contractor may not use or disclose it exce...
1234,17,30,29,permission,prohibition,0.380952,0.600000,0.333333,0.0,1.0,h. The contractor may disclose VA sensitive in...,The contractor may not use or disclose it exce...


In [123]:
# Cell 79 — Distribution of important signals

print("Modality conflicts:")
print(dev_pair_df["modality_conflict"].value_counts())

print("\nProposition Jaccard:")
print(dev_pair_df["proposition_jaccard"].describe())

print("\nCondition similarity:")
print(dev_pair_df["condition_similarity"].describe())

print("\nParty similarity:")
print(dev_pair_df["party_similarity"].describe())

Modality conflicts:
modality_conflict
0    8828
1    1542
Name: count, dtype: int64

Proposition Jaccard:
count    10370.000000
mean         0.106337
std          0.115252
min          0.000000
25%          0.037037
50%          0.083333
75%          0.142857
max          1.000000
Name: proposition_jaccard, dtype: float64

Condition similarity:
count    10370.000000
mean         0.645162
std          0.478465
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: condition_similarity, dtype: float64

Party similarity:
count    10370.000000
mean         0.525339
std          0.444838
min          0.000000
25%          0.000000
50%          0.666667
75%          1.000000
max          1.000000
Name: party_similarity, dtype: float64


In [124]:
# Cell 80 — Strong modality conflicts with high proposition similarity

display(
    dev_pair_df[
        (dev_pair_df["modality_conflict"] == 1) &
        (dev_pair_df["proposition_similarity"] >= 0.5)
    ][
        [
            "document_id",
            "span_a",
            "span_b",
            "proposition_similarity",
            "proposition_overlap",
            "proposition_jaccard",
            "condition_similarity",
            "party_similarity",
            "modality_a",
            "modality_b",
            "text_a",
            "text_b"
        ]
    ]
    .sort_values(
        "proposition_similarity",
        ascending=False
    )
    .head(20)
)

,document_id,span_a,span_b,proposition_similarity,proposition_overlap,proposition_jaccard,condition_similarity,party_similarity,modality_a,modality_b,text_a,text_b
7147,590,11,7,0.5,1.0,0.75,1.0,1.0,prohibition,obligation,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7240,590,7,11,0.5,1.0,0.75,1.0,1.0,obligation,prohibition,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7260,590,11,7,0.5,1.0,0.75,1.0,1.0,prohibition,obligation,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."


In [128]:
def extract_proposition_parts(text):
    text = normalize_clause(text)

    if not isinstance(text, str):
        text = str(text)

    # Start with the actual clause text
    cleaned = text

    # Remove common legal boilerplate / conditions
    cleaned = re.sub(
        r"\b(?:provided that|notwithstanding|subject to|in the event that)\b.*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # Remove modality words
    action_text = re.sub(
        r"\b(?:shall not|shall|must not|must|may not|may|will not|will|"
        r"cannot|is required to|are required to|is permitted to|"
        r"are permitted to)\b",
        " ",
        cleaned,
        flags=re.IGNORECASE
    )

    action_text = re.sub(r"\s+", " ", action_text).strip()

    tokens = re.findall(
        r"\b[a-zA-Z]{3,}\b",
        action_text.lower()
    )

    if not tokens:
        return {
            "subject": "",
            "action": "",
            "object": ""
        }

    # Recognized legal parties/entities
    party_patterns = [
        "receiving party",
        "disclosing party",
        "company",
        "contractor",
        "buyer",
        "seller",
        "vendor",
        "customer",
        "employer",
        "employee",
        "party",
        "parties"
    ]

    subject = ""

    for party in party_patterns:
        if party in action_text.lower():
            subject = party
            break

    # Common legal action verbs
    action_verbs = {
        "disclose", "discloses", "disclosing",
        "use", "uses", "using",
        "receive", "receives", "receiving",
        "provide", "provides", "providing",
        "return", "returns", "returning",
        "destroy", "destroys", "destroying",
        "retain", "retains", "retaining",
        "assign", "assigns", "assigning",
        "transfer", "transfers", "transferring",
        "notify", "notifies", "notifying",
        "pay", "pays", "paying",
        "deliver", "delivers", "delivering",
        "terminate", "terminates", "terminating",
        "modify", "modifies", "modifying",
        "reverse", "reverse-engineer"
    }

    action = ""
    action_index = None

    for i, token in enumerate(tokens):
        if token in action_verbs:
            action = token
            action_index = i
            break

    # Everything after the action is treated as the object.
    if action_index is not None:
        object_tokens = tokens[action_index + 1:]
    else:
        object_tokens = tokens

    object_tokens = [
        t for t in object_tokens
        if t not in {
            "the",
            "and",
            "or",
            "any",
            "such",
            "information",
            "agreement",
            "foregoing"
        }
    ]

    return {
        "subject": subject,
        "action": action,
        "object": " ".join(object_tokens[:12])
    }

In [129]:
parts_a = dev_pair_df["text_a"].apply(
    extract_proposition_parts
)

parts_b = dev_pair_df["text_b"].apply(
    extract_proposition_parts
)

dev_pair_df["subject_a"] = parts_a.apply(
    lambda x: x["subject"]
)

dev_pair_df["action_a"] = parts_a.apply(
    lambda x: x["action"]
)

dev_pair_df["object_a"] = parts_a.apply(
    lambda x: x["object"]
)

dev_pair_df["subject_b"] = parts_b.apply(
    lambda x: x["subject"]
)

dev_pair_df["action_b"] = parts_b.apply(
    lambda x: x["action"]
)

dev_pair_df["object_b"] = parts_b.apply(
    lambda x: x["object"]
)

display(
    dev_pair_df[
        [
            "text_a",
            "subject_a",
            "action_a",
            "object_a",
            "text_b",
            "subject_b",
            "action_b",
            "object_b"
        ]
    ].head(20)
)

,text_a,subject_a,action_a,object_a,text_b,subject_b,action_b,object_b
0,"Upon the Disclosing Party’s written request, t...",receiving party,disclosing,party written request receiving party promptly...,"Additionally, nothing in this CNDA constitutes...",disclosing party,disclosing,party accuracy confidential
1,"Upon the Disclosing Party’s written request, t...",receiving party,disclosing,party written request receiving party promptly...,(d) information that the Disclosing Party auth...,receiving party,disclosing,party authorizes receiving party disclose
2,"Upon the Disclosing Party’s written request, t...",receiving party,disclosing,party written request receiving party promptly...,The Receiving Party acknowledges and agrees th...,receiving party,receiving,party acknowledges agrees that has entered int...
3,"Upon the Disclosing Party’s written request, t...",receiving party,disclosing,party written request receiving party promptly...,ARTICLE 2- CONFIDENTIAL INFORMATION,,,article confidential
4,"Additionally, nothing in this CNDA constitutes...",disclosing party,disclosing,party accuracy confidential,(d) information that the Disclosing Party auth...,receiving party,disclosing,party authorizes receiving party disclose
5,"Additionally, nothing in this CNDA constitutes...",disclosing party,disclosing,party accuracy confidential,The Receiving Party acknowledges and agrees th...,receiving party,receiving,party acknowledges agrees that has entered int...
6,"Additionally, nothing in this CNDA constitutes...",disclosing party,disclosing,party accuracy confidential,ARTICLE 2- CONFIDENTIAL INFORMATION,,,article confidential
7,(d) information that the Disclosing Party auth...,receiving party,disclosing,party authorizes receiving party disclose,The Receiving Party acknowledges and agrees th...,receiving party,receiving,party acknowledges agrees that has entered int...
8,(d) information that the Disclosing Party auth...,receiving party,disclosing,party authorizes receiving party disclose,ARTICLE 2- CONFIDENTIAL INFORMATION,,,article confidential
9,The Receiving Party acknowledges and agrees th...,receiving party,receiving,party acknowledges agrees that has entered int...,ARTICLE 2- CONFIDENTIAL INFORMATION,,,article confidential


In [132]:
def normalize_proposition(text):
    if not isinstance(text, str):
        text = str(text)

    text = normalize_clause(text).lower()

    # Remove common legal boilerplate
    boilerplate = [
        r"\bthe foregoing\b",
        r"\bnotwithstanding the foregoing\b",
        r"\bwithout limiting the foregoing\b",
        r"\bfor the avoidance of doubt\b",
        r"\bin addition\b",
        r"\bprovided that\b",
        r"\bsubject to\b",
        r"\bin the event that\b",
        r"\bexcept as set forth in\b",
        r"\bexcept as provided in\b",
        r"\bwith respect to\b",
        r"\bon behalf of\b"
    ]

    for pattern in boilerplate:
        text = re.sub(pattern, " ", text)

    # Remove modality
    text = re.sub(
        r"\b(?:shall not|must not|may not|will not|"
        r"shall|must|may|will|cannot|"
        r"is required to|are required to|"
        r"is permitted to|are permitted to)\b",
        " ",
        text
    )

    # Remove common legal filler
    filler = {
        "the", "a", "an", "any", "such", "this", "that",
        "herein", "thereof", "thereto", "therein",
        "foregoing", "agreement", "section", "article",
        "party", "parties"
    }

    tokens = re.findall(r"\b[a-z]{3,}\b", text)

    tokens = [
        token for token in tokens
        if token not in filler
    ]

    return " ".join(tokens)

In [133]:
dev_pair_df["normalized_prop_a"] = (
    dev_pair_df["text_a"]
    .apply(normalize_proposition)
)

dev_pair_df["normalized_prop_b"] = (
    dev_pair_df["text_b"]
    .apply(normalize_proposition)
)

display(
    dev_pair_df[
        [
            "text_a",
            "normalized_prop_a",
            "text_b",
            "normalized_prop_b"
        ]
    ].head(20)
)

,text_a,normalized_prop_a,text_b,normalized_prop_b
0,"Upon the Disclosing Party’s written request, t...",upon disclosing written request receiving prom...,"Additionally, nothing in this CNDA constitutes...",additionally nothing cnda constitutes warranty...
1,"Upon the Disclosing Party’s written request, t...",upon disclosing written request receiving prom...,(d) information that the Disclosing Party auth...,information disclosing authorizes receiving di...
2,"Upon the Disclosing Party’s written request, t...",upon disclosing written request receiving prom...,The Receiving Party acknowledges and agrees th...,receiving acknowledges and agrees has entered ...
3,"Upon the Disclosing Party’s written request, t...",upon disclosing written request receiving prom...,ARTICLE 2- CONFIDENTIAL INFORMATION,confidential information
4,"Additionally, nothing in this CNDA constitutes...",additionally nothing cnda constitutes warranty...,(d) information that the Disclosing Party auth...,information disclosing authorizes receiving di...
5,"Additionally, nothing in this CNDA constitutes...",additionally nothing cnda constitutes warranty...,The Receiving Party acknowledges and agrees th...,receiving acknowledges and agrees has entered ...
6,"Additionally, nothing in this CNDA constitutes...",additionally nothing cnda constitutes warranty...,ARTICLE 2- CONFIDENTIAL INFORMATION,confidential information
7,(d) information that the Disclosing Party auth...,information disclosing authorizes receiving di...,The Receiving Party acknowledges and agrees th...,receiving acknowledges and agrees has entered ...
8,(d) information that the Disclosing Party auth...,information disclosing authorizes receiving di...,ARTICLE 2- CONFIDENTIAL INFORMATION,confidential information
9,The Receiving Party acknowledges and agrees th...,receiving acknowledges and agrees has entered ...,ARTICLE 2- CONFIDENTIAL INFORMATION,confidential information


In [134]:
def core_proposition_similarity(a, b):
    tokens_a = set(
        re.findall(r"\b[a-z]{3,}\b", str(a))
    )

    tokens_b = set(
        re.findall(r"\b[a-z]{3,}\b", str(b))
    )

    if not tokens_a or not tokens_b:
        return 0.0

    intersection = len(tokens_a & tokens_b)
    union = len(tokens_a | tokens_b)

    return intersection / union


dev_pair_df["core_prop_similarity"] = (
    dev_pair_df.apply(
        lambda r: core_proposition_similarity(
            r["normalized_prop_a"],
            r["normalized_prop_b"]
        ),
        axis=1
    )
)

print(
    dev_pair_df["core_prop_similarity"].describe()
)

count    10370.000000
mean         0.105230
std          0.108287
min          0.000000
25%          0.035714
50%          0.085106
75%          0.142857
max          1.000000
Name: core_prop_similarity, dtype: float64


In [135]:
strong_alignment = dev_pair_df[
    (dev_pair_df["modality_conflict"] == 1) &
    (dev_pair_df["core_prop_similarity"] >= 0.30)
].sort_values(
    "core_prop_similarity",
    ascending=False
)

print(
    "Strong aligned modality conflicts:",
    len(strong_alignment)
)

display(
    strong_alignment[
        [
            "document_id",
            "span_a",
            "span_b",
            "modality_a",
            "modality_b",
            "core_prop_similarity",
            "normalized_prop_a",
            "normalized_prop_b",
            "text_a",
            "text_b"
        ]
    ].head(20)
)

Strong aligned modality conflicts: 19


,document_id,span_a,span_b,modality_a,modality_b,core_prop_similarity,normalized_prop_a,normalized_prop_b,text_a,text_b
7260,590,11,7,prohibition,obligation,1.000000,proprietary information include information,proprietary information include,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7240,590,7,11,obligation,prohibition,1.000000,proprietary information include,proprietary information include information,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7147,590,11,7,prohibition,obligation,1.000000,proprietary information include information,proprietary information include,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
3870,405,20,24,prohibition,obligation,0.454545,you and your representatives use evaluation ma...,without limiting generality possible transacti...,You and your representatives will not use any ...,Without limiting the generality of the foregoi...
6810,582,13,21,prohibition,permission,0.430769,you and company also each agree itself its aff...,event you your representatives become legally ...,"You and the Company also each agree, on behalf...",4. In the event you or any of your Representat...
8438,64,47,59,obligation,prohibition,0.421053,information kept confidential perpetuity and s...,retained information held receiving and kept c...,The Information shall be kept confidential in ...,The Retained Information shall be held by the ...
8333,64,59,47,prohibition,obligation,0.421053,retained information held receiving and kept c...,information kept confidential perpetuity and s...,The Retained Information shall be held by the ...,The Information shall be kept confidential in ...
361,13,43,111,permission,prohibition,0.363636,disclosing disclose discuss confidential infor...,disclosure confidential information nature obl...,A Disclosing Party may disclose or discuss a P...,Disclosure of Confidential Information of any ...
8439,64,61,59,obligation,prohibition,0.350000,and all oral information continue held strict ...,retained information held receiving and kept c...,Any and all oral Information shall continue to...,The Retained Information shall be held by the ...
384,13,43,62,permission,prohibition,0.333333,disclosing disclose discuss confidential infor...,disclose confidential information third withou...,A Disclosing Party may disclose or discuss a P...,Parties shall not disclose Confidential Inform...


In [136]:
def scope_compatibility(row):
    cond_a = set(row["conditions_a"]) if isinstance(row["conditions_a"], list) else set()
    cond_b = set(row["conditions_b"]) if isinstance(row["conditions_b"], list) else set()

    # Both unconditional
    if not cond_a and not cond_b:
        return 1.0

    # One conditional, one unconditional
    if not cond_a or not cond_b:
        return 0.5

    # Same / overlapping conditions
    overlap = len(cond_a & cond_b)

    if overlap > 0:
        return 1.0

    return 0.0

In [137]:
def get_clause_features(document_id, span_id):
    rows = structured_clauses[
        (structured_clauses["document_id"].astype(str) == str(document_id)) &
        (structured_clauses["span_id"] == int(span_id))
    ]

    if rows.empty:
        return {
            "conditions": [],
            "temporal": [],
            "numbers": [],
            "parties": []
        }

    row = rows.iloc[0]

    return {
        "conditions": row["conditions"],
        "temporal": row["temporal"],
        "numbers": row["numbers"],
        "parties": row["parties"]
    }


features_a = dev_pair_df.apply(
    lambda r: get_clause_features(
        r["document_id"],
        r["span_a"]
    ),
    axis=1
)

features_b = dev_pair_df.apply(
    lambda r: get_clause_features(
        r["document_id"],
        r["span_b"]
    ),
    axis=1
)

dev_pair_df["conditions_a"] = features_a.apply(
    lambda x: x["conditions"]
)

dev_pair_df["conditions_b"] = features_b.apply(
    lambda x: x["conditions"]
)

dev_pair_df["temporal_a"] = features_a.apply(
    lambda x: x["temporal"]
)

dev_pair_df["temporal_b"] = features_b.apply(
    lambda x: x["temporal"]
)

dev_pair_df["numbers_a"] = features_a.apply(
    lambda x: x["numbers"]
)

dev_pair_df["numbers_b"] = features_b.apply(
    lambda x: x["numbers"]
)

In [140]:
dev_pair_df["scope_compatibility"] = dev_pair_df.apply(
    scope_compatibility,
    axis=1
)

print(
    dev_pair_df["scope_compatibility"].value_counts(
        normalize=True
    )
)

scope_compatibility
1.0    0.645130
0.5    0.312247
0.0    0.042623
Name: proportion, dtype: float64


In [141]:
strong_alignment = dev_pair_df[
    (dev_pair_df["modality_conflict"] == 1) &
    (dev_pair_df["core_prop_similarity"] >= 0.30)
].copy()

display(
    strong_alignment[
        [
            "document_id",
            "span_a",
            "span_b",
            "modality_a",
            "modality_b",
            "core_prop_similarity",
            "scope_compatibility",
            "conditions_a",
            "conditions_b",
            "temporal_a",
            "temporal_b",
            "numbers_a",
            "numbers_b",
            "text_a",
            "text_b"
        ]
    ].sort_values(
        "core_prop_similarity",
        ascending=False
    )
)

,document_id,span_a,span_b,modality_a,modality_b,core_prop_similarity,scope_compatibility,conditions_a,conditions_b,temporal_a,temporal_b,numbers_a,numbers_b,text_a,text_b
7260,590,11,7,prohibition,obligation,1.000000,1.0,[],[],[],[],[],[],Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7240,590,7,11,obligation,prohibition,1.000000,1.0,[],[],[],[],[],[],"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7147,590,11,7,prohibition,obligation,1.000000,1.0,[],[],[],[],[],[],Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
3870,405,20,24,prohibition,obligation,0.454545,0.5,[],[in the event that a Possible Transaction is n...,[],[],[],[],You and your representatives will not use any ...,Without limiting the generality of the foregoi...
6810,582,13,21,prohibition,permission,0.430769,1.0,[with respect to any such possible Transaction],[with respect to any such possible Transaction],[],[],[],[4],"You and the Company also each agree, on behalf...",4. In the event you or any of your Representat...
8438,64,47,59,obligation,prohibition,0.421053,0.5,[],[unless the Receiving Party is prohibited from...,[],[],[],[],The Information shall be kept confidential in ...,The Retained Information shall be held by the ...
8333,64,59,47,prohibition,obligation,0.421053,0.5,[unless the Receiving Party is prohibited from...,[],[],[],[],[],The Retained Information shall be held by the ...,The Information shall be kept confidential in ...
361,13,43,111,permission,prohibition,0.363636,1.0,[],[],[],[],[],[],A Disclosing Party may disclose or discuss a P...,Disclosure of Confidential Information of any ...
8439,64,61,59,obligation,prohibition,0.350000,0.5,[],[unless the Receiving Party is prohibited from...,[],[],[],[],Any and all oral Information shall continue to...,The Retained Information shall be held by the ...
384,13,43,62,permission,prohibition,0.333333,1.0,[],[],[],[],[],[],A Disclosing Party may disclose or discuss a P...,Parties shall not disclose Confidential Inform...


In [142]:
EXCEPTION_PATTERNS = [
    r"\bunless\b",
    r"\bexcept\b",
    r"\bexcept as\b",
    r"\bexcept that\b",
    r"\bprovided that\b",
    r"\bsubject to\b",
    r"\bnotwithstanding\b",
    r"\bin the event that\b",
    r"\bto the extent that\b",
    r"\bif\b",
    r"\bonly if\b"
]

def detect_exception(text):
    if not isinstance(text, str):
        text = str(text)

    text = text.lower()

    return int(
        any(
            re.search(pattern, text)
            for pattern in EXCEPTION_PATTERNS
        )
    )


dev_pair_df["exception_a"] = (
    dev_pair_df["text_a"].apply(detect_exception)
)

dev_pair_df["exception_b"] = (
    dev_pair_df["text_b"].apply(detect_exception)
)

print(
    dev_pair_df[
        ["exception_a", "exception_b"]
    ].value_counts()
)

exception_a  exception_b
0            0              6754
             1              1714
1            0              1482
             1               420
Name: count, dtype: int64


In [143]:
dev_pair_df["scope_aware_score"] = (
    0.45 * dev_pair_df["core_prop_similarity"]
    + 0.20 * dev_pair_df["proposition_jaccard"]
    + 0.20 * dev_pair_df["modality_conflict"]
    + 0.15 * dev_pair_df["party_similarity"]
)

# No modality conflict -> strongly suppress
dev_pair_df.loc[
    dev_pair_df["modality_conflict"] == 0,
    "scope_aware_score"
] *= 0.10

# Poor proposition alignment -> suppress
dev_pair_df.loc[
    dev_pair_df["core_prop_similarity"] < 0.30,
    "scope_aware_score"
] *= 0.10

# Exceptions / conditions require additional reasoning.
# Do NOT completely eliminate these cases.
dev_pair_df.loc[
    (dev_pair_df["exception_a"] == 1) |
    (dev_pair_df["exception_b"] == 1),
    "scope_aware_score"
] *= 0.60

print(
    dev_pair_df["scope_aware_score"].describe()
)

count    10370.000000
mean         0.007447
std          0.026610
min          0.000000
25%          0.000590
50%          0.001400
75%          0.002534
max          0.950000
Name: scope_aware_score, dtype: float64


In [144]:
def classify_conflict(row):
    
    if row["modality_conflict"] == 0:
        return "not_conflicting"

    if row["core_prop_similarity"] < 0.30:
        return "different_proposition"

    if row["exception_a"] or row["exception_b"]:
        return "scope_dependent_review"

    if row["core_prop_similarity"] >= 0.70:
        return "strong_conflict"

    return "potential_conflict"


dev_pair_df["conflict_category"] = (
    dev_pair_df.apply(
        classify_conflict,
        axis=1
    )
)

print(
    dev_pair_df["conflict_category"].value_counts()
)

conflict_category
not_conflicting           8828
different_proposition     1523
scope_dependent_review       9
potential_conflict           7
strong_conflict              3
Name: count, dtype: int64


In [145]:
display(
    dev_pair_df[
        dev_pair_df["conflict_category"] != "not_conflicting"
    ][
        [
            "document_id",
            "span_a",
            "span_b",
            "conflict_category",
            "core_prop_similarity",
            "proposition_jaccard",
            "modality_a",
            "modality_b",
            "exception_a",
            "exception_b",
            "conditions_a",
            "conditions_b",
            "text_a",
            "text_b"
        ]
    ]
    .sort_values(
        [
            "conflict_category",
            "core_prop_similarity"
        ],
        ascending=[True, False]
    )
    .head(50)
)

,document_id,span_a,span_b,conflict_category,core_prop_similarity,proposition_jaccard,modality_a,modality_b,exception_a,exception_b,conditions_a,conditions_b,text_a,text_b
1296,17,22,28,different_proposition,0.281250,0.285714,obligation,prohibition,0,1,[],[except to the extent necessary to perform the...,"d. The contractor, the contractor's employees,...",g. Any information that the contractor and its...
1337,17,22,28,different_proposition,0.281250,0.285714,obligation,prohibition,0,1,[],[except to the extent necessary to perform the...,"d. The contractor, the contractor's employees,...",g. Any information that the contractor and its...
11,10,34,39,different_proposition,0.277778,0.333333,obligation,prohibition,1,1,[except as set forth in section 2],[Except as set forth in Section 2],"In addition, except as set forth in section 2....","Except as set forth in Section 2.06, the Recei..."
4099,411,26,25,different_proposition,0.277778,0.333333,obligation,prohibition,1,1,[In the event that such protective order or ot...,[unless notice is prohibited by law) so that t...,In the event that such protective order or oth...,(c) In the event that a Stockholder is request...
4159,411,25,26,different_proposition,0.277778,0.333333,prohibition,obligation,1,1,[unless notice is prohibited by law) so that t...,[In the event that such protective order or ot...,(c) In the event that a Stockholder is request...,In the event that such protective order or oth...
4199,411,25,26,different_proposition,0.277778,0.333333,prohibition,obligation,1,1,[unless notice is prohibited by law) so that t...,[In the event that such protective order or ot...,(c) In the event that a Stockholder is request...,In the event that such protective order or oth...
4209,411,25,26,different_proposition,0.277778,0.333333,prohibition,obligation,1,1,[unless notice is prohibited by law) so that t...,[In the event that such protective order or ot...,(c) In the event that a Stockholder is request...,In the event that such protective order or oth...
4246,411,25,26,different_proposition,0.277778,0.333333,prohibition,obligation,1,1,[unless notice is prohibited by law) so that t...,[In the event that such protective order or ot...,(c) In the event that a Stockholder is request...,In the event that such protective order or oth...
1164,16,12,14,different_proposition,0.269231,0.320000,obligation,prohibition,0,1,[],[except for those copies required for use by A...,Access to or disclosure of Confidential Inform...,4. Recipient shall not make copies of the Conf...
9031,71,9,11,different_proposition,0.269231,0.260870,prohibition,obligation,0,0,[],[],Prospect and Prospect’s Broker acknowledge tha...,In the event the transaction is not successful...


In [146]:
def get_gold_span_set(document_id, label_id):
    entry = document_map[str(document_id)]
    document = entry["document"]

    for annotation_set in document.get("annotation_sets", []):
        annotations = annotation_set.get("annotations", {})
        annotation = annotations.get(str(label_id))

        if annotation is not None:
            return {
                int(s)
                for s in annotation.get("spans", [])
                if str(s).isdigit()
            }

    return set()


dev_pair_df["gold_spans"] = dev_pair_df.apply(
    lambda r: get_gold_span_set(
        r["document_id"],
        r["label_id"]
    ),
    axis=1
)

print(
    "Pairs with gold evidence:",
    dev_pair_df["gold_spans"].apply(
        lambda x: len(x) > 0
    ).sum()
)

Pairs with gold evidence: 6140


In [147]:
def evidence_hit(row):
    gold = row["gold_spans"]

    if not gold:
        return 0

    return int(
        row["span_a"] in gold or
        row["span_b"] in gold
    )


dev_pair_df["evidence_hit"] = (
    dev_pair_df.apply(
        evidence_hit,
        axis=1
    )
)

print(
    "Candidate pairs touching gold evidence:",
    dev_pair_df["evidence_hit"].sum()
)

Candidate pairs touching gold evidence: 1838


In [148]:
strong_candidates = dev_pair_df[
    dev_pair_df["conflict_category"] == "strong_conflict"
].copy()

potential_candidates = dev_pair_df[
    dev_pair_df["conflict_category"] == "potential_conflict"
].copy()

print(
    "Strong candidates:",
    len(strong_candidates)
)

print(
    "Strong candidates hitting gold evidence:",
    strong_candidates["evidence_hit"].sum()
)

print(
    "Potential candidates:",
    len(potential_candidates)
)

print(
    "Potential candidates hitting gold evidence:",
    potential_candidates["evidence_hit"].sum()
)

Strong candidates: 3
Strong candidates hitting gold evidence: 0
Potential candidates: 7
Potential candidates hitting gold evidence: 2


In [149]:
display(
    strong_candidates[
        [
            "document_id",
            "label_id",
            "span_a",
            "span_b",
            "core_prop_similarity",
            "proposition_jaccard",
            "modality_a",
            "modality_b",
            "conditions_a",
            "conditions_b",
            "gold_spans",
            "evidence_hit",
            "text_a",
            "text_b"
        ]
    ]
)

,document_id,label_id,span_a,span_b,core_prop_similarity,proposition_jaccard,modality_a,modality_b,conditions_a,conditions_b,gold_spans,evidence_hit,text_a,text_b
7147,590,nda-1,11,7,1.0,0.75,prohibition,obligation,[],[],{},0,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7240,590,nda-2,7,11,1.0,0.75,obligation,prohibition,[],[],{6},0,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7260,590,nda-3,11,7,1.0,0.75,prohibition,obligation,[],[],{6},0,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."


In [150]:
dev_evidence_eval = dev_pair_df[
    (dev_pair_df["target"] == 2) &
    (dev_pair_df["gold_spans"].apply(lambda x: len(x) > 0))
].copy()

print(
    "Contradiction examples with gold evidence:",
    dev_evidence_eval[
        ["document_id", "label_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Total pair rows:",
    len(dev_evidence_eval)
)

Contradiction examples with gold evidence: 95
Total pair rows: 950


In [152]:
gold_evidence_by_example = (
    dev_evidence_eval
    .groupby(
        ["document_id", "label_id"]
    )["gold_spans"]
    .first()
    .reset_index()
)

retrieved_spans_by_example = (
    dev_evidence_eval
    .groupby(
        ["document_id", "label_id"]
    )["span_a"]
    .apply(set)
    .reset_index()
)

span_b_sets = (
    dev_evidence_eval
    .groupby(
        ["document_id", "label_id"]
    )["span_b"]
    .apply(set)
    .reset_index()
)

# Correct way to combine two sets
retrieved_spans_by_example["retrieved_spans"] = [
    a.union(b)
    for a, b in zip(
        retrieved_spans_by_example["span_a"],
        span_b_sets["span_b"]
    )
]

retrieved_spans_by_example = retrieved_spans_by_example[
    [
        "document_id",
        "label_id",
        "retrieved_spans"
    ]
]

evidence_eval = gold_evidence_by_example.merge(
    retrieved_spans_by_example,
    on=["document_id", "label_id"]
)

evidence_eval["evidence_recall"] = evidence_eval.apply(
    lambda r: (
        len(
            r["gold_spans"] &
            r["retrieved_spans"]
        ) /
        len(r["gold_spans"])
    ),
    axis=1
)

print(
    "Mean evidence recall:",
    round(
        evidence_eval["evidence_recall"].mean(),
        4
    )
)

print(
    "Examples with at least one gold span retrieved:",
    (
        evidence_eval["evidence_recall"] > 0
    ).sum(),
    "/",
    len(evidence_eval)
)

Mean evidence recall: 0.3553
Examples with at least one gold span retrieved: 49 / 95


In [153]:
structured_eval = dev_pair_df[
    (dev_pair_df["target"] == 2) &
    (dev_pair_df["gold_spans"].apply(lambda x: len(x) > 0)) &
    (dev_pair_df["conflict_category"].isin([
        "strong_conflict",
        "potential_conflict",
        "scope_dependent_review"
    ]))
].copy()

structured_hits = (
    structured_eval
    .groupby(
        ["document_id", "label_id"]
    )["evidence_hit"]
    .max()
    .reset_index()
)

print(
    "Contradiction examples detected by structured candidates:",
    len(structured_hits)
)

print(
    "Detected examples touching gold evidence:",
    structured_hits["evidence_hit"].sum()
)

Contradiction examples detected by structured candidates: 1
Detected examples touching gold evidence: 0


In [154]:
display(
    dev_pair_df[
        (dev_pair_df["target"] == 2) &
        (dev_pair_df["gold_spans"].apply(lambda x: len(x) > 0)) &
        (dev_pair_df["conflict_category"].isin([
            "strong_conflict",
            "potential_conflict",
            "scope_dependent_review"
        ]))
    ][
        [
            "document_id",
            "label_id",
            "span_a",
            "span_b",
            "conflict_category",
            "core_prop_similarity",
            "modality_a",
            "modality_b",
            "gold_spans",
            "evidence_hit",
            "text_a",
            "text_b"
        ]
    ]
    .sort_values(
        ["evidence_hit", "core_prop_similarity"],
        ascending=[False, False]
    )
    .head(30)
)

,document_id,label_id,span_a,span_b,conflict_category,core_prop_similarity,modality_a,modality_b,gold_spans,evidence_hit,text_a,text_b
7240,590,nda-2,7,11,strong_conflict,1.0,obligation,prohibition,{6},0,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...


In [155]:
dev_pair_df["clause_reasoning_score_a"] = (
    0.40 * dev_pair_df["core_prop_similarity"]
    + 0.20 * dev_pair_df["proposition_jaccard"]
    + 0.15 * dev_pair_df["party_similarity"]
    + 0.15 * dev_pair_df["condition_similarity"]
    + 0.10 * dev_pair_df["modality_conflict"]
)

dev_pair_df["clause_reasoning_score_b"] = (
    0.40 * dev_pair_df["core_prop_similarity"]
    + 0.20 * dev_pair_df["proposition_jaccard"]
    + 0.15 * dev_pair_df["party_similarity"]
    + 0.15 * dev_pair_df["condition_similarity"]
    + 0.10 * dev_pair_df["modality_conflict"]
)

In [156]:
pair_scores_a = dev_pair_df[
    [
        "document_id",
        "label_id",
        "span_a",
        "clause_reasoning_score_a"
    ]
].rename(
    columns={
        "span_a": "span_id",
        "clause_reasoning_score_a": "reasoning_score"
    }
)

pair_scores_b = dev_pair_df[
    [
        "document_id",
        "label_id",
        "span_b",
        "clause_reasoning_score_b"
    ]
].rename(
    columns={
        "span_b": "span_id",
        "clause_reasoning_score_b": "reasoning_score"
    }
)

clause_reasoning_scores = pd.concat(
    [pair_scores_a, pair_scores_b],
    ignore_index=True
)

clause_reasoning_scores = (
    clause_reasoning_scores
    .groupby(
        ["document_id", "label_id", "span_id"],
        as_index=False
    )["reasoning_score"]
    .max()
)

print(
    "Unique clause candidates:",
    len(clause_reasoning_scores)
)

Unique clause candidates: 5185


In [157]:
retrieval_info = pd.concat(
    [
        dev_pair_df[
            [
                "document_id",
                "label_id",
                "span_a",
                "retrieval_a"
            ]
        ].rename(
            columns={
                "span_a": "span_id",
                "retrieval_a": "retrieval_score"
            }
        ),

        dev_pair_df[
            [
                "document_id",
                "label_id",
                "span_b",
                "retrieval_b"
            ]
        ].rename(
            columns={
                "span_b": "span_id",
                "retrieval_b": "retrieval_score"
            }
        )
    ],
    ignore_index=True
)

retrieval_info = (
    retrieval_info
    .groupby(
        ["document_id", "label_id", "span_id"],
        as_index=False
    )["retrieval_score"]
    .max()
)

clause_reasoning_scores = clause_reasoning_scores.merge(
    retrieval_info,
    on=[
        "document_id",
        "label_id",
        "span_id"
    ],
    how="left"
)

display(
    clause_reasoning_scores.head()
)

,document_id,label_id,span_id,reasoning_score,retrieval_score
0,10,nda-1,22,0.278889,0.126756
1,10,nda-1,30,0.355294,0.149521
2,10,nda-1,44,0.355294,0.178320
3,10,nda-1,49,0.350809,0.137766
4,10,nda-1,90,0.350000,0.168049


In [158]:
clause_reasoning_scores["rerank_score"] = (
    0.60 * clause_reasoning_scores["retrieval_score"]
    + 0.40 * clause_reasoning_scores["reasoning_score"]
)

print(
    clause_reasoning_scores["rerank_score"].describe()
)

count    5185.000000
mean        0.196490
std         0.067403
min         0.022926
25%         0.152152
50%         0.188527
75%         0.230135
max         0.562971
Name: rerank_score, dtype: float64


In [159]:
def evaluate_top_k_clause_recall(
    score_df,
    k
):
    results = []

    for (document_id, label_id), group in score_df.groupby(
        ["document_id", "label_id"]
    ):
        gold = get_gold_span_set(
            document_id,
            label_id
        )

        if not gold:
            continue

        ranked = group.sort_values(
            "rerank_score",
            ascending=False
        ).head(k)

        predicted = set(
            ranked["span_id"].astype(int)
        )

        hit = int(
            len(gold & predicted) > 0
        )

        results.append(hit)

    return np.mean(results), len(results)


for k in [1, 3, 5]:
    recall, n = evaluate_top_k_clause_recall(
        clause_reasoning_scores,
        k
    )

    print(
        f"Reranked Recall@{k}: "
        f"{recall:.4f} "
        f"({recall*100:.2f}%), "
        f"n={n}"
    )

Reranked Recall@1: 0.2443 (24.43%), n=614
Reranked Recall@3: 0.4723 (47.23%), n=614
Reranked Recall@5: 0.6580 (65.80%), n=614


In [160]:
import os

save_dir = "/kaggle/working/lexconflict_saved"
os.makedirs(save_dir, exist_ok=True)

objects_to_save = {
    "ml_df": ml_df,
    "clauses_df": clauses_df,
    "structured_clauses": structured_clauses,
    "dev_structured_df": dev_structured_df,
    "dev_pair_df": dev_pair_df,
    "clause_reasoning_scores": clause_reasoning_scores
}

for name, obj in objects_to_save.items():
    obj.to_pickle(f"{save_dir}/{name}.pkl")

print("Everything saved successfully.")
print("\nFiles:")
for f in os.listdir(save_dir):
    print("-", f)

Everything saved successfully.

Files:
- structured_clauses.pkl
- dev_pair_df.pkl
- clause_reasoning_scores.pkl
- dev_structured_df.pkl
- clauses_df.pkl
- ml_df.pkl


In [1]:
import os

DATA_PATH = "/kaggle/input/datasets/aditik1234/legal-data"

print("Saved files:")
for root, dirs, files in os.walk(DATA_PATH):
    for f in files:
        print(os.path.join(root, f))

Saved files:
/kaggle/input/datasets/aditik1234/legal-data/legaldatawhatever/structured_clauses.pkl
/kaggle/input/datasets/aditik1234/legal-data/legaldatawhatever/clause_reasoning_scores.pkl
/kaggle/input/datasets/aditik1234/legal-data/legaldatawhatever/ml_df.pkl
/kaggle/input/datasets/aditik1234/legal-data/legaldatawhatever/clauses_df.pkl
/kaggle/input/datasets/aditik1234/legal-data/legaldatawhatever/dev_pair_df.pkl
/kaggle/input/datasets/aditik1234/legal-data/legaldatawhatever/dev_structured_df.pkl


In [2]:
import os
import pickle
import pandas as pd

BASE = "/kaggle/input/datasets/aditik1234/legal-data/legaldatawhatever"

# Load saved data
with open(os.path.join(BASE, "structured_clauses.pkl"), "rb") as f:
    structured_clauses = pickle.load(f)

with open(os.path.join(BASE, "clause_reasoning_scores.pkl"), "rb") as f:
    clause_reasoning_scores = pickle.load(f)

with open(os.path.join(BASE, "ml_df.pkl"), "rb") as f:
    ml_df = pickle.load(f)

with open(os.path.join(BASE, "clauses_df.pkl"), "rb") as f:
    clauses_df = pickle.load(f)

with open(os.path.join(BASE, "dev_pair_df.pkl"), "rb") as f:
    dev_pair_df = pickle.load(f)

with open(os.path.join(BASE, "dev_structured_df.pkl"), "rb") as f:
    dev_structured_df = pickle.load(f)

print("Loaded successfully!")

print("clauses_df:", clauses_df.shape)
print("structured_clauses:", structured_clauses.shape)
print("ml_df:", ml_df.shape)
print("dev_pair_df:", dev_pair_df.shape)
print("dev_structured_df:", dev_structured_df.shape)

Loaded successfully!
clauses_df: (47321, 3)
structured_clauses: (47321, 9)
ml_df: (10319, 10)
dev_pair_df: (10370, 48)
dev_structured_df: (5185, 14)


In [3]:
print("CLAUSES")
print(clauses_df.columns.tolist())

print("\nSTRUCTURED CLAUSES")
print(structured_clauses.columns.tolist())

print("\nML DATA")
print(ml_df.columns.tolist())

print("\nDEV PAIRS")
print(dev_pair_df.columns.tolist())

print("\nDEV STRUCTURED")
print(dev_structured_df.columns.tolist())

CLAUSES
['document_id', 'span_id', 'text']

STRUCTURED CLAUSES
['document_id', 'span_id', 'text', 'modality', 'negation', 'temporal', 'numbers', 'parties', 'conditions']

ML DATA
['document_id', 'split', 'label_id', 'choice', 'target', 'hypothesis', 'document_text', 'span_ids', 'retrieved_context', 'bert_text']

DEV PAIRS
['document_id', 'label_id', 'target', 'span_a', 'span_b', 'text_a', 'text_b', 'retrieval_a', 'retrieval_b', 'modality_a', 'modality_b', 'modality_conflict', 'negation_conflict', 'proposition_similarity', 'condition_similarity', 'party_similarity', 'structured_conflict_score', 'proposition_overlap', 'strict_conflict_score', 'proposition_jaccard', 'aligned_conflict', 'aligned_conflict_score', 'subject_a', 'action_a', 'object_a', 'subject_b', 'action_b', 'object_b', 'proposition_alignment', 'structured_conflict_candidate', 'normalized_prop_a', 'normalized_prop_b', 'core_prop_similarity', 'conditions_a', 'conditions_b', 'temporal_a', 'temporal_b', 'numbers_a', 'numbers_b'

In [5]:
print(ml_df["split"].value_counts())

print("\nTargets:")
print(ml_df["target"].value_counts())

print("\nChoices:")
print(ml_df["choice"].value_counts())

split
train    7191
test     2091
dev      1037
Name: count, dtype: int64

Targets:
target
1    5017
0    4146
2    1156
Name: count, dtype: int64

Choices:
choice
Entailment       5017
NotMentioned     4146
Contradiction    1156
Name: count, dtype: int64


In [6]:
def extract_gold_spans(row):
    """
    Return gold evidence span IDs for one ML example.
    Handles missing/empty annotation information safely.
    """
    
    span_ids = row.get("span_ids", [])
    
    if isinstance(span_ids, str):
        try:
            span_ids = eval(span_ids)
        except:
            span_ids = []
    
    if span_ids is None:
        span_ids = []
    
    return set(span_ids)


ml_df["gold_spans"] = ml_df.apply(
    extract_gold_spans,
    axis=1
)

print(
    "Examples with gold evidence:",
    (ml_df["gold_spans"].apply(len) > 0).sum()
)

print(
    "Examples without gold evidence:",
    (ml_df["gold_spans"].apply(len) == 0).sum()
)

Examples with gold evidence: 6173
Examples without gold evidence: 4146


In [8]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Fix gold span representation
# ---------------------------------------------------------

def parse_span_ids(x):
    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    if isinstance(x, float) and pd.isna(x):
        return set()

    if isinstance(x, str):
        try:
            value = eval(x)
            if isinstance(value, (list, tuple, set)):
                return set(value)
        except:
            pass

    return set()


# Make a normalized gold-span column
ml_dev = ml_df[ml_df["split"] == "dev"].copy()

ml_dev["gold_spans_set"] = (
    ml_dev["span_ids"]
    .apply(parse_span_ids)
)

# Keep only the columns we need
gold_by_example = (
    ml_dev[
        ["document_id", "label_id", "gold_spans_set"]
    ]
    .copy()
)

# IMPORTANT:
# Do NOT drop_duplicates() on a set column.
# Deduplicate using document_id + label_id only.
gold_by_example = (
    gold_by_example
    .groupby(["document_id", "label_id"], as_index=False)
    ["gold_spans_set"]
    .first()
)

# ---------------------------------------------------------
# Prepare retrieved structured clauses
# ---------------------------------------------------------

dev_structured_eval = dev_structured_df.merge(
    gold_by_example,
    on=["document_id", "label_id"],
    how="left"
)

dev_structured_eval["gold_spans_set"] = (
    dev_structured_eval["gold_spans_set"]
    .apply(parse_span_ids)
)

dev_structured_eval["span_id_set"] = (
    dev_structured_eval["span_id"]
    .apply(lambda x: {x})
)

dev_structured_eval["gold_hit"] = (
    dev_structured_eval.apply(
        lambda r:
        len(
            r["span_id_set"] &
            r["gold_spans_set"]
        ) > 0,
        axis=1
    )
)

print(
    "Retrieved clauses:",
    len(dev_structured_eval)
)

print(
    "Clauses hitting gold evidence:",
    dev_structured_eval["gold_hit"].sum()
)

Retrieved clauses: 5185
Clauses hitting gold evidence: 480


In [9]:
# ---------------------------------------------------------
# Structured evidence recall
# ---------------------------------------------------------

structured_recall = (
    dev_structured_eval
    .groupby(["document_id", "label_id"])["gold_hit"]
    .max()
)

print(
    "Structured Evidence Recall@5:",
    round(structured_recall.mean(), 4)
)

print(
    "Examples with retrieved gold evidence:",
    structured_recall.sum(),
    "/",
    len(structured_recall)
)

Structured Evidence Recall@5: 0.3896
Examples with retrieved gold evidence: 404 / 1037


In [25]:
# =========================================================
# STEP: Build document-level novelty/conflict scores
# =========================================================

df = dev_pair_df.copy()

# Make sure numeric columns are numeric
score_cols = [
    "scope_aware_score",
    "core_prop_similarity",
    "proposition_jaccard",
    "modality_conflict",
    "negation_conflict",
    "condition_similarity",
    "party_similarity",
    "aligned_conflict_score",
    "clause_reasoning_score_a",
    "clause_reasoning_score_b"
]

for col in score_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        ).fillna(0)


# ---------------------------------------------------------
# Strong evidence score
# ---------------------------------------------------------

df["final_conflict_score"] = (
    0.30 * df["core_prop_similarity"] +
    0.20 * df["proposition_jaccard"] +
    0.20 * df["scope_aware_score"] +
    0.15 * df["aligned_conflict_score"] +
    0.10 * df["modality_conflict"] +
    0.05 * df["negation_conflict"]
)


# ---------------------------------------------------------
# Suppress weak semantic matches
# ---------------------------------------------------------

df.loc[
    df["core_prop_similarity"] < 0.25,
    "final_conflict_score"
] *= 0.25


# ---------------------------------------------------------
# Strong modality conflict
# ---------------------------------------------------------

df["strong_modality_conflict"] = (
    (df["modality_conflict"] == 1) &
    (df["core_prop_similarity"] >= 0.30) &
    (df["scope_aware_score"] >= 0.50)
)


# ---------------------------------------------------------
# Strong structured conflict
# ---------------------------------------------------------

df["strong_conflict"] = (
    (df["final_conflict_score"] >= 0.50) &
    (
        df["strong_modality_conflict"] |
        (df["aligned_conflict_score"] >= 0.50)
    )
)


print(
    "Strong conflict pairs:",
    df["strong_conflict"].sum()
)

Strong conflict pairs: 8


In [11]:
# =========================================================
# Aggregate pair-level evidence to example level
# =========================================================

example_scores = (
    df.groupby(
        ["document_id", "label_id", "target"]
    )
    .agg(
        max_conflict_score=(
            "final_conflict_score",
            "max"
        ),

        mean_conflict_score=(
            "final_conflict_score",
            "mean"
        ),

        strong_conflict_count=(
            "strong_conflict",
            "sum"
        ),

        modality_conflict_count=(
            "modality_conflict",
            "sum"
        ),

        max_core_similarity=(
            "core_prop_similarity",
            "max"
        ),

        max_scope_score=(
            "scope_aware_score",
            "max"
        )
    )
    .reset_index()
)

print(example_scores.shape)

display(
    example_scores.sort_values(
        "max_conflict_score",
        ascending=False
    ).head(20)
)

(1037, 9)


,document_id,label_id,target,max_conflict_score,mean_conflict_score,strong_conflict_count,modality_conflict_count,max_core_similarity,max_scope_score
714,590,nda-1,0,0.904375,0.104510,1,2,1.000000,0.950000
726,590,nda-3,1,0.904375,0.143938,1,3,1.000000,0.950000
724,590,nda-2,2,0.904375,0.112503,1,4,1.000000,0.950000
681,582,nda-10,1,0.581687,0.083499,1,4,0.430769,0.629560
36,13,nda-11,0,0.531136,0.132028,1,4,0.363636,0.586364
948,73,nda-4,1,0.531000,0.137707,0,1,1.000000,0.080000
940,73,nda-15,1,0.531000,0.130615,0,0,1.000000,0.080000
939,73,nda-13,1,0.531000,0.133958,0,0,1.000000,0.080000
938,73,nda-12,1,0.531000,0.134402,0,0,1.000000,0.080000
937,73,nda-11,0,0.531000,0.176198,0,0,1.000000,0.080000


In [12]:
# =========================================================
# Contradiction detection diagnostic
# =========================================================

example_scores["gold_conflict"] = (
    example_scores["target"] == 2
)

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:

    example_scores["pred_conflict"] = (
        example_scores["max_conflict_score"]
        >= threshold
    )

    tp = (
        example_scores["pred_conflict"] &
        example_scores["gold_conflict"]
    ).sum()

    fp = (
        example_scores["pred_conflict"] &
        ~example_scores["gold_conflict"]
    ).sum()

    fn = (
        ~example_scores["pred_conflict"] &
        example_scores["gold_conflict"]
    ).sum()

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    print(
        f"Threshold={threshold:.2f} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | "
        f"F1={f1:.4f}"
    )

Threshold=0.20 | Precision=0.1189 | Recall=0.2316 | F1=0.1571
Threshold=0.30 | Precision=0.0957 | Recall=0.0947 | F1=0.0952
Threshold=0.40 | Precision=0.0714 | Recall=0.0316 | F1=0.0438
Threshold=0.50 | Precision=0.1250 | Recall=0.0316 | F1=0.0504
Threshold=0.60 | Precision=0.3333 | Recall=0.0105 | F1=0.0204
Threshold=0.70 | Precision=0.3333 | Recall=0.0105 | F1=0.0204


In [13]:
# ============================================================
# NEXT STEP: BUILD A BETTER STRUCTURED EVIDENCE CANDIDATE SET
# ============================================================

import pandas as pd
import numpy as np

print("dev_structured_df:", dev_structured_df.shape)
print("dev_pair_df:", dev_pair_df.shape)
print("ml_df:", ml_df.shape)

# ------------------------------------------------------------
# 1. Keep only useful columns from structured retrieval
# ------------------------------------------------------------

structured_base = dev_structured_df[
    [
        "document_id",
        "label_id",
        "hypothesis",
        "target",
        "span_id",
        "clause",
        "retrieval_score",
        "modality",
        "negation",
        "conditions",
        "temporal",
        "numbers",
        "parties",
        "proposition"
    ]
].copy()

# Remove accidental duplicates
structured_base = structured_base.drop_duplicates(
    subset=["document_id", "label_id", "span_id"]
)

print("Unique structured clauses:", len(structured_base))

dev_structured_df: (5185, 15)
dev_pair_df: (10370, 48)
ml_df: (10319, 11)
Unique structured clauses: 5185


In [14]:
# ============================================================
# 2. NORMALIZE GOLD SPANS SAFELY
# ============================================================

def normalize_span_set(x):

    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    if isinstance(x, float) and np.isnan(x):
        return set()

    if isinstance(x, str):
        try:
            value = eval(x)

            if isinstance(value, set):
                return value

            if isinstance(value, (list, tuple)):
                return set(value)

        except:
            pass

    return set()


gold_df = ml_df[
    ml_df["split"] == "dev"
][
    [
        "document_id",
        "label_id",
        "span_ids"
    ]
].copy()

gold_df["gold_spans"] = gold_df["span_ids"].apply(
    normalize_span_set
)

gold_df = gold_df[
    [
        "document_id",
        "label_id",
        "gold_spans"
    ]
].drop_duplicates(
    subset=["document_id", "label_id"]
)

print("Dev examples with gold evidence:", len(gold_df))

Dev examples with gold evidence: 1037


In [15]:
# ============================================================
# 3. CHECK GOLD-SPAN COVERAGE BEFORE CONFLICT DETECTION
# ============================================================

retrieved_span_sets = (
    structured_base
    .groupby(
        ["document_id", "label_id"]
    )["span_id"]
    .apply(set)
    .reset_index(name="retrieved_spans")
)

coverage_df = gold_df.merge(
    retrieved_span_sets,
    on=["document_id", "label_id"],
    how="left"
)

coverage_df["retrieved_spans"] = coverage_df[
    "retrieved_spans"
].apply(normalize_span_set)

coverage_df["gold_retrieved"] = coverage_df.apply(
    lambda r: len(
        r["gold_spans"] &
        r["retrieved_spans"]
    ),
    axis=1
)

coverage_df["gold_recall"] = coverage_df.apply(
    lambda r:
        len(r["gold_spans"] & r["retrieved_spans"])
        / len(r["gold_spans"])
        if len(r["gold_spans"]) > 0
        else 0,
    axis=1
)

print(
    "Examples with at least one gold span:",
    (coverage_df["gold_retrieved"] > 0).sum(),
    "/",
    len(coverage_df)
)

print(
    "Mean structured gold coverage:",
    round(coverage_df["gold_recall"].mean(), 4)
)

Examples with at least one gold span: 404 / 1037
Mean structured gold coverage: 0.2762


In [16]:
# ============================================================
# 4. EXPAND STRUCTURED RETRIEVAL TO TOP-10
# ============================================================

structured_top10 = (
    dev_structured_df
    .sort_values(
        ["document_id", "label_id", "retrieval_score"],
        ascending=[True, True, False]
    )
    .groupby(
        ["document_id", "label_id"],
        group_keys=False
    )
    .head(10)
    .copy()
)

print(
    "Structured top-10 rows:",
    len(structured_top10)
)

print(
    "Unique examples:",
    structured_top10[
        ["document_id", "label_id"]
    ].drop_duplicates().shape[0]
)

Structured top-10 rows: 5185
Unique examples: 1037


In [17]:
# ============================================================
# 5. TOP-10 EVIDENCE COVERAGE
# ============================================================

top10_span_sets = (
    structured_top10
    .groupby(
        ["document_id", "label_id"]
    )["span_id"]
    .apply(set)
    .reset_index(name="retrieved_spans")
)

top10_eval = gold_df.merge(
    top10_span_sets,
    on=["document_id", "label_id"],
    how="left"
)

top10_eval["retrieved_spans"] = top10_eval[
    "retrieved_spans"
].apply(normalize_span_set)

top10_eval["gold_hit"] = top10_eval.apply(
    lambda r:
        len(
            r["gold_spans"] &
            r["retrieved_spans"]
        ) > 0,
    axis=1
)

print(
    "Top-10 examples hitting gold:",
    top10_eval["gold_hit"].sum(),
    "/",
    len(top10_eval)
)

print(
    "Top-10 evidence recall:",
    round(
        top10_eval["gold_hit"].mean(),
        4
    )
)

Top-10 examples hitting gold: 404 / 1037
Top-10 evidence recall: 0.3896


In [18]:
# ============================================================
# 6. BUILD STRUCTURED PAIRS FROM TOP-10
# ============================================================

from itertools import combinations

structured_pairs = []

for (document_id, label_id), group in structured_top10.groupby(
    ["document_id", "label_id"]
):

    rows = group.to_dict("records")

    # Compare every pair within the top-10
    for a, b in combinations(rows, 2):

        structured_pairs.append({
            "document_id": document_id,
            "label_id": label_id,

            "target": a.get("target"),

            "span_a": a["span_id"],
            "span_b": b["span_id"],

            "text_a": a["clause"],
            "text_b": b["clause"],

            "retrieval_a": a["retrieval_score"],
            "retrieval_b": b["retrieval_score"],

            "modality_a": a["modality"],
            "modality_b": b["modality"],

            "negation_a": a["negation"],
            "negation_b": b["negation"],

            "conditions_a": a["conditions"],
            "conditions_b": b["conditions"],

            "temporal_a": a["temporal"],
            "temporal_b": b["temporal"],

            "numbers_a": a["numbers"],
            "numbers_b": b["numbers"],

            "parties_a": a["parties"],
            "parties_b": b["parties"],

            "proposition_a": a["proposition"],
            "proposition_b": b["proposition"]
        })


structured_pairs_df = pd.DataFrame(
    structured_pairs
)

print(
    "Structured pair count:",
    len(structured_pairs_df)
)

print(
    "Unique examples:",
    structured_pairs_df[
        ["document_id", "label_id"]
    ].drop_duplicates().shape[0]
)

Structured pair count: 10370
Unique examples: 1037


In [19]:
import os
import pickle
import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/aditik1234/legal-data/legaldatawhatever"

def load_pickle(name):
    path = os.path.join(BASE, name)
    with open(path, "rb") as f:
        return pickle.load(f)

clauses_df = load_pickle("clauses_df.pkl")
structured_clauses = load_pickle("structured_clauses.pkl")
ml_df = load_pickle("ml_df.pkl")
dev_pair_df = load_pickle("dev_pair_df.pkl")
dev_structured_df = load_pickle("dev_structured_df.pkl")
clause_reasoning_scores = load_pickle("clause_reasoning_scores.pkl")

print("Loaded successfully")
print("clauses_df:", clauses_df.shape)
print("structured_clauses:", structured_clauses.shape)
print("ml_df:", ml_df.shape)
print("dev_pair_df:", dev_pair_df.shape)
print("dev_structured_df:", dev_structured_df.shape)

Loaded successfully
clauses_df: (47321, 3)
structured_clauses: (47321, 9)
ml_df: (10319, 10)
dev_pair_df: (10370, 48)
dev_structured_df: (5185, 14)


In [20]:
df = dev_pair_df.copy()

# Make sure numeric columns are numeric
score_cols = [
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "modality_conflict",
    "clause_reasoning_score_a",
    "clause_reasoning_score_b"
]

for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Reasoning confidence
df["reasoning_confidence"] = (
    df["clause_reasoning_score_a"] +
    df["clause_reasoning_score_b"]
) / 2

# Final conflict/evidence score
df["final_conflict_score"] = (
    0.30 * df["core_prop_similarity"] +
    0.20 * df["proposition_jaccard"] +
    0.15 * df["scope_compatibility"] +
    0.20 * df["modality_conflict"] +
    0.15 * df["reasoning_confidence"]
)

print(df["final_conflict_score"].describe())

count    10370.000000
mean         0.240835
std          0.109758
min          0.000000
25%          0.172500
50%          0.222206
75%          0.295629
max          0.942500
Name: final_conflict_score, dtype: float64


In [21]:
df["candidate_conflict"] = (
    (df["final_conflict_score"] >= 0.30) &
    (
        (df["core_prop_similarity"] >= 0.35) |
        (df["proposition_jaccard"] >= 0.30)
    ) &
    (df["modality_conflict"] == 1)
)

print(
    "Final conflict candidates:",
    df["candidate_conflict"].sum()
)

Final conflict candidates: 28


In [22]:
def to_set(x):
    if isinstance(x, set):
        return x
    if isinstance(x, (list, tuple)):
        return set(x)
    if pd.isna(x):
        return set()
    return set()

df["gold_span_set"] = df["gold_spans"].apply(to_set)

df["pair_span_set"] = df.apply(
    lambda r: {r["span_a"], r["span_b"]},
    axis=1
)

df["gold_evidence_hit"] = df.apply(
    lambda r: len(r["pair_span_set"] & r["gold_span_set"]) > 0,
    axis=1
)

print(
    "Pairs touching gold evidence:",
    df["gold_evidence_hit"].sum()
)

Pairs touching gold evidence: 1838


In [23]:
pred = df["candidate_conflict"]
gold = (
    (df["target"] == 2) &
    df["gold_evidence_hit"]
)

tp = ((pred == 1) & (gold == 1)).sum()
fp = ((pred == 1) & (gold == 0)).sum()
fn = ((pred == 0) & (gold == 1)).sum()

precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)

print(f"\nPrecision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

TP: 0
FP: 28
FN: 208

Precision: 0.0000
Recall:    0.0000
F1:        0.0000


In [28]:
# ============================================================
# NEXT STEP: GOLD-AWARE CONFLICT SCORE DIAGNOSTIC
# ============================================================

import pandas as pd
import numpy as np

df = dev_pair_df.copy()

# ------------------------------------------------------------
# 1. Make sure conflict categories are available
# ------------------------------------------------------------

if "conflict_category" not in df.columns:
    print("ERROR: conflict_category is missing")
else:
    print("Conflict categories:")
    print(df["conflict_category"].value_counts(dropna=False))


# ------------------------------------------------------------
# 2. Identify whether each pair touches gold evidence
# ------------------------------------------------------------

def parse_span_set(x):
    if isinstance(x, set):
        return x
    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    try:
        if pd.isna(x):
            return set()
    except:
        pass

    if isinstance(x, str):
        try:
            value = eval(x)
            if isinstance(value, (list, tuple, set)):
                return set(value)
        except:
            pass

    return set()


df["gold_span_set"] = df["gold_spans"].apply(parse_span_set)

df["a_is_gold"] = df.apply(
    lambda r: r["span_a"] in r["gold_span_set"],
    axis=1
)

df["b_is_gold"] = df.apply(
    lambda r: r["span_b"] in r["gold_span_set"],
    axis=1
)

df["pair_touches_gold"] = (
    df["a_is_gold"] | df["b_is_gold"]
)


# ------------------------------------------------------------
# 3. Gold contradiction pair
# ------------------------------------------------------------

# A pair is a gold contradiction pair when BOTH:
#   - it is supported by gold evidence
#   - the pair is labelled as an actual contradiction/conflict
#
# We use the target/choice information already attached
# to the pair dataframe.

if "target" in df.columns:
    print("\nTarget distribution:")
    print(df["target"].value_counts(dropna=False))

if "aligned_conflict" in df.columns:
    print("\nAligned conflict distribution:")
    print(df["aligned_conflict"].value_counts(dropna=False))


# ------------------------------------------------------------
# 4. Inspect the highest scoring pairs
# ------------------------------------------------------------

cols = [
    "document_id",
    "label_id",
    "span_a",
    "span_b",
    "target",
    "modality_a",
    "modality_b",
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "final_conflict_score",
    "pair_touches_gold",
    "text_a",
    "text_b"
]

cols = [c for c in cols if c in df.columns]

display(
    df.sort_values(
        "final_conflict_score",
        ascending=False
    )[cols].head(30)
)


# ------------------------------------------------------------
# 5. Compare scores for gold vs non-gold pairs
# ------------------------------------------------------------

print("\nFINAL CONFLICT SCORE BY GOLD CONTACT:")
print(
    df.groupby("pair_touches_gold")["final_conflict_score"]
      .agg(["count", "mean", "max"])
)


# ------------------------------------------------------------
# 6. Most important diagnostic:
#    Are the gold pairs actually receiving high scores?
# ------------------------------------------------------------

gold_pairs = df[df["pair_touches_gold"]].copy()
non_gold_pairs = df[~df["pair_touches_gold"]].copy()

print("\nGold-touching pairs:", len(gold_pairs))
print("Non-gold pairs:", len(non_gold_pairs))

print("\nGold-touching score statistics:")
print(
    gold_pairs["final_conflict_score"].describe()
)

print("\nNon-gold score statistics:")
print(
    non_gold_pairs["final_conflict_score"].describe()
)


# ------------------------------------------------------------
# 7. Top gold-touching pairs
# ------------------------------------------------------------

print("\nTOP GOLD-TOUCHING PAIRS:")
display(
    gold_pairs.sort_values(
        "final_conflict_score",
        ascending=False
    )[cols].head(20)
)

Conflict categories:
conflict_category
not_conflicting           8828
different_proposition     1523
scope_dependent_review       9
potential_conflict           7
strong_conflict              3
Name: count, dtype: int64

Target distribution:
target
1    5190
0    4230
2     950
Name: count, dtype: int64

Aligned conflict distribution:
aligned_conflict
0    10342
1       28
Name: count, dtype: int64


,document_id,label_id,span_a,span_b,target,modality_a,modality_b,core_prop_similarity,proposition_jaccard,scope_compatibility,strict_conflict_score,final_conflict_score,pair_touches_gold,text_a,text_b
7240,590,nda-2,7,11,2,obligation,prohibition,1.000000,0.750000,1.0,0.900000,0.883750,False,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7260,590,nda-3,11,7,1,prohibition,obligation,1.000000,0.750000,1.0,0.900000,0.883750,False,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7147,590,nda-1,11,7,0,prohibition,obligation,1.000000,0.750000,1.0,0.900000,0.883750,False,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
6810,582,nda-10,13,21,1,prohibition,permission,0.430769,0.428571,1.0,0.702336,0.703390,True,"You and the Company also each agree, on behalf...",4. In the event you or any of your Representat...
361,13,nda-11,43,111,0,permission,prohibition,0.363636,0.363636,1.0,0.630952,0.652197,False,A Disclosing Party may disclose or discuss a P...,Disclosure of Confidential Information of any ...
4689,440,nda-19,19,7,0,prohibition,obligation,0.200000,0.400000,1.0,0.866667,0.632733,False,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
4647,440,nda-15,19,7,1,prohibition,obligation,0.200000,0.400000,1.0,0.866667,0.632733,False,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
9113,71,nda-2,9,10,0,prohibition,obligation,0.322581,0.321429,1.0,0.585606,0.622972,False,Prospect and Prospect’s Broker acknowledge tha...,Prospect’s and Prospect’s Broker’s disseminati...
9133,71,nda-3,9,10,0,prohibition,obligation,0.322581,0.321429,1.0,0.585606,0.622972,False,Prospect and Prospect’s Broker acknowledge tha...,Prospect’s and Prospect’s Broker’s disseminati...
9160,71,nda-7,10,9,1,obligation,prohibition,0.322581,0.321429,1.0,0.585606,0.622972,True,Prospect’s and Prospect’s Broker’s disseminati...,Prospect and Prospect’s Broker acknowledge tha...



FINAL CONFLICT SCORE BY GOLD CONTACT:
                   count      mean      max
pair_touches_gold                          
False               8532  0.091109  0.88375
True                1838  0.092294  0.70339

Gold-touching pairs: 1838
Non-gold pairs: 8532

Gold-touching score statistics:
count    1838.000000
mean        0.092294
std         0.095923
min         0.000000
25%         0.015156
50%         0.027855
75%         0.167128
max         0.703390
Name: final_conflict_score, dtype: float64

Non-gold score statistics:
count    8532.000000
mean        0.091109
std         0.097205
min         0.000000
25%         0.012774
50%         0.027070
75%         0.169742
max         0.883750
Name: final_conflict_score, dtype: float64

TOP GOLD-TOUCHING PAIRS:


,document_id,label_id,span_a,span_b,target,modality_a,modality_b,core_prop_similarity,proposition_jaccard,scope_compatibility,strict_conflict_score,final_conflict_score,pair_touches_gold,text_a,text_b
6810,582,nda-10,13,21,1,prohibition,permission,0.430769,0.428571,1.0,0.702336,0.703390,True,"You and the Company also each agree, on behalf...",4. In the event you or any of your Representat...
9160,71,nda-7,10,9,1,obligation,prohibition,0.322581,0.321429,1.0,0.585606,0.622972,True,Prospect’s and Prospect’s Broker’s disseminati...,Prospect and Prospect’s Broker acknowledge tha...
3870,405,nda-4,20,24,1,prohibition,obligation,0.454545,0.428571,0.5,0.618769,0.564258,True,You and your representatives will not use any ...,Without limiting the generality of the foregoi...
11,10,nda-10,34,39,1,obligation,prohibition,0.277778,0.333333,0.0,0.641667,0.525183,True,"In addition, except as set forth in section 2....","Except as set forth in Section 2.06, the Recei..."
9790,79,nda-2,10,6,2,prohibition,obligation,0.250000,0.272727,1.0,0.857143,0.475166,True,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
7954,61,nda-4,25,23,1,prohibition,obligation,0.045455,0.055556,1.0,0.817391,0.454120,True,3. Company shall not,Company also agrees that it shall treat the Co...
1164,16,nda-5,12,14,1,obligation,prohibition,0.269231,0.320000,0.5,0.543634,0.440583,True,Access to or disclosure of Confidential Inform...,4. Recipient shall not make copies of the Conf...
8383,64,nda-15,86,52,1,prohibition,obligation,0.190476,0.210526,1.0,0.672667,0.408932,True,The provisions of this Confidentiality Agreeme...,The Receiving Party shall be responsible for a...
7525,598,nda-13,4,15,1,prohibition,obligation,0.200000,0.250000,1.0,0.650000,0.400925,True,Confidential Information shall not include inf...,3. Use of Confidential Information: The Recipi...
7610,598,nda-4,15,4,1,obligation,prohibition,0.200000,0.250000,1.0,0.650000,0.400925,True,3. Use of Confidential Information: The Recipi...,Confidential Information shall not include inf...


In [27]:
# ============================================================
# FIX: RECREATE FINAL CONFLICT SCORE
# ============================================================

import pandas as pd
import numpy as np

df = dev_pair_df.copy()

# ------------------------------------------------------------
# Check required columns
# ------------------------------------------------------------

required = [
    "strict_conflict_score",
    "aligned_conflict_score",
    "scope_aware_score",
    "core_prop_similarity",
    "modality_conflict",
    "negation_conflict"
]

print("Available score columns:")
for c in required:
    print(c, "->", c in df.columns)

# ------------------------------------------------------------
# Safely create missing numeric columns
# ------------------------------------------------------------

for c in required:
    if c not in df.columns:
        df[c] = 0.0

    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    ).fillna(0.0)

# ------------------------------------------------------------
# Recreate final conflict score
# ------------------------------------------------------------

df["final_conflict_score"] = (
    0.35 * df["strict_conflict_score"]
    + 0.30 * df["aligned_conflict_score"]
    + 0.20 * df["scope_aware_score"]
    + 0.10 * df["modality_conflict"]
    + 0.05 * df["negation_conflict"]
)

print("\nFinal conflict score created.")

print(
    df["final_conflict_score"].describe()
)

# Put it back into dev_pair_df

dev_pair_df = df

Available score columns:
strict_conflict_score -> True
aligned_conflict_score -> True
scope_aware_score -> True
core_prop_similarity -> True
modality_conflict -> True
negation_conflict -> True

Final conflict score created.
count    10370.000000
mean         0.091319
std          0.096976
min          0.000000
25%          0.013252
50%          0.027215
75%          0.169608
max          0.883750
Name: final_conflict_score, dtype: float64


In [29]:
label_map = {
    0: "NotMentioned",
    1: "Entailment",
    2: "Contradiction"
}

print(ml_df["target"].value_counts().sort_index())
print("\nLabel mapping:")
for k, v in label_map.items():
    print(k, "=", v)

target
0    4146
1    5017
2    1156
Name: count, dtype: int64

Label mapping:
0 = NotMentioned
1 = Entailment
2 = Contradiction


In [30]:
example_labels = (
    ml_df[
        ml_df["split"].isin(["train", "dev", "test"])
    ][
        ["document_id", "label_id", "target", "choice"]
    ]
    .drop_duplicates(
        subset=["document_id", "label_id"]
    )
)

print("Unique examples:", len(example_labels))

print(
    example_labels["target"]
    .value_counts()
    .sort_index()
)

Unique examples: 10319
target
0    4146
1    5017
2    1156
Name: count, dtype: int64


In [31]:
dev_labels = (
    example_labels[
        example_labels["document_id"].isin(
            dev_pair_df["document_id"].unique()
        )
    ]
    .copy()
)

print("Dev examples:", len(dev_labels))

Dev examples: 1037


In [32]:
pair_features = [
    "strict_conflict_score",
    "aligned_conflict_score",
    "scope_aware_score",
    "core_prop_similarity",
    "proposition_jaccard",
    "proposition_overlap",
    "condition_similarity",
    "party_similarity",
    "modality_conflict",
    "negation_conflict",
    "scope_compatibility",
    "clause_reasoning_score_a",
    "clause_reasoning_score_b"
]

pair_features = [
    c for c in pair_features
    if c in dev_pair_df.columns
]

print("Using features:")
print(pair_features)

Using features:
['strict_conflict_score', 'aligned_conflict_score', 'scope_aware_score', 'core_prop_similarity', 'proposition_jaccard', 'proposition_overlap', 'condition_similarity', 'party_similarity', 'modality_conflict', 'negation_conflict', 'scope_compatibility', 'clause_reasoning_score_a', 'clause_reasoning_score_b']


In [33]:
dev_example_features = (
    dev_pair_df
    .groupby(["document_id", "label_id"])
    .agg(
        **{
            f"{c}_max": (c, "max")
            for c in pair_features
        },
        **{
            f"{c}_mean": (c, "mean")
            for c in pair_features
        }
    )
    .reset_index()
)

print(
    "Example-level feature shape:",
    dev_example_features.shape
)

Example-level feature shape: (1037, 28)


In [34]:
conflict_counts = (
    dev_pair_df
    .groupby(["document_id", "label_id"])
    .agg(
        modality_conflict_count=(
            "modality_conflict",
            "sum"
        ),
        aligned_conflict_count=(
            "aligned_conflict",
            "sum"
        ),
        structured_candidate_count=(
            "structured_conflict_candidate",
            "sum"
        ),
        pair_count=("span_a", "count")
    )
    .reset_index()
)

dev_example_features = dev_example_features.merge(
    conflict_counts,
    on=["document_id", "label_id"],
    how="left"
)

print(dev_example_features.shape)

(1037, 32)


In [35]:
dev_example_features = dev_example_features.merge(
    dev_labels[
        ["document_id", "label_id", "target", "choice"]
    ],
    on=["document_id", "label_id"],
    how="inner"
)

print(
    dev_example_features["target"]
    .value_counts()
    .sort_index()
)

display(
    dev_example_features.head()
)

target
0    423
1    519
2     95
Name: count, dtype: int64


,document_id,label_id,strict_conflict_score_max,aligned_conflict_score_max,scope_aware_score_max,core_prop_similarity_max,proposition_jaccard_max,proposition_overlap_max,condition_similarity_max,party_similarity_max,...,negation_conflict_mean,scope_compatibility_mean,clause_reasoning_score_a_mean,clause_reasoning_score_b_mean,modality_conflict_count,aligned_conflict_count,structured_candidate_count,pair_count,target,choice
0,10,nda-1,0.500000,0.003917,0.002097,0.222222,0.200000,0.666667,1.0,1.0,...,0.0,1.0,0.283908,0.283908,0,0,0,10,2,Contradiction
1,10,nda-10,0.641667,0.483333,0.028000,0.277778,0.333333,0.800000,1.0,1.0,...,0.3,0.4,0.260243,0.260243,3,1,0,10,1,Entailment
2,10,nda-11,0.471579,0.033500,0.036214,0.200000,0.195652,0.800000,1.0,1.0,...,0.3,0.6,0.271371,0.271371,3,0,0,10,0,NotMentioned
3,10,nda-12,0.783333,0.076667,0.073333,1.000000,0.666667,1.000000,1.0,1.0,...,0.0,0.8,0.262117,0.262117,0,0,0,10,0,NotMentioned
4,10,nda-13,0.783333,0.076667,0.073333,1.000000,0.666667,1.000000,1.0,1.0,...,0.0,1.0,0.331133,0.331133,0,0,0,10,1,Entailment


In [36]:
dev_example_features["is_contradiction"] = (
    dev_example_features["target"] == 2
).astype(int)

print(
    "Contradiction examples:",
    dev_example_features["is_contradiction"].sum()
)

print(
    "Non-contradiction examples:",
    (
        dev_example_features["is_contradiction"] == 0
    ).sum()
)

Contradiction examples: 95
Non-contradiction examples: 942


In [37]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

feature_cols = [
    c for c in dev_example_features.columns
    if c not in [
        "document_id",
        "label_id",
        "target",
        "choice",
        "is_contradiction"
    ]
]

X_dev = dev_example_features[feature_cols]
y_dev = dev_example_features["is_contradiction"]

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

model.fit(X_dev, y_dev)

dev_example_features["conflict_probability"] = (
    model.predict_proba(X_dev)[:, 1]
)

print(
    dev_example_features[
        "conflict_probability"
    ].describe()
)

count    1037.000000
mean        0.273088
std         0.150720
min         0.009621
25%         0.158817
50%         0.245233
75%         0.351445
max         0.816513
Name: conflict_probability, dtype: float64


In [38]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in np.arange(0.05, 1.00, 0.05):

    pred = (
        dev_example_features["conflict_probability"]
        >= threshold
    ).astype(int)

    precision = precision_score(
        y_dev,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_dev,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_dev,
        pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.sort_values(
        "f1",
        ascending=False
    )
)

,threshold,precision,recall,f1
9,0.50,0.833333,0.947368,0.886700
8,0.45,0.646259,1.000000,0.785124
10,0.55,0.910448,0.642105,0.753086
7,0.40,0.477387,1.000000,0.646259
6,0.35,0.361217,1.000000,0.530726
11,0.60,0.921053,0.368421,0.526316
5,0.30,0.260989,1.000000,0.413943
4,0.25,0.188119,1.000000,0.316667
12,0.65,1.000000,0.178947,0.303571
3,0.20,0.143072,1.000000,0.250329


In [39]:
best_threshold = (
    threshold_df
    .sort_values("f1", ascending=False)
    .iloc[0]["threshold"]
)

print("Best Dev threshold:", best_threshold)

Best Dev threshold: 0.5


In [40]:
dev_pred = (
    dev_example_features["conflict_probability"]
    >= best_threshold
).astype(int)

print("Classification Report:")
print(
    classification_report(
        y_dev,
        dev_pred,
        target_names=[
            "Non-Contradiction",
            "Contradiction"
        ],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_dev,
        dev_pred
    )
)

Classification Report:
                   precision    recall  f1-score   support

Non-Contradiction       0.99      0.98      0.99       942
    Contradiction       0.83      0.95      0.89        95

         accuracy                           0.98      1037
        macro avg       0.91      0.96      0.94      1037
     weighted avg       0.98      0.98      0.98      1037

Confusion Matrix:
[[924  18]
 [  5  90]]


In [41]:
detected = (
    dev_example_features[
        dev_example_features["conflict_probability"]
        >= best_threshold
    ]
    .sort_values(
        "conflict_probability",
        ascending=False
    )
)

print(
    "Detected contradiction examples:",
    len(detected)
)

display(
    detected[
        [
            "document_id",
            "label_id",
            "target",
            "choice",
            "conflict_probability",
            "strict_conflict_score_max",
            "aligned_conflict_score_max",
            "scope_aware_score_max",
            "core_prop_similarity_max",
            "proposition_jaccard_max",
            "modality_conflict_count",
            "aligned_conflict_count"
        ]
    ].head(30)
)

Detected contradiction examples: 108


,document_id,label_id,target,choice,conflict_probability,strict_conflict_score_max,aligned_conflict_score_max,scope_aware_score_max,core_prop_similarity_max,proposition_jaccard_max,modality_conflict_count,aligned_conflict_count
435,435,nda-2,2,Contradiction,0.816513,0.453529,0.004317,0.002650,0.250000,0.200000,0,0
639,56,nda-2,2,Contradiction,0.743717,0.425000,0.004500,0.003069,0.285714,0.285714,0,0
945,73,nda-2,2,Contradiction,0.729902,0.598214,0.067714,0.046286,0.428571,0.600000,1,0
673,575,nda-2,2,Contradiction,0.716901,0.634848,0.065000,0.047500,0.500000,0.500000,1,0
486,45,nda-2,2,Contradiction,0.711225,0.333271,0.004186,0.002594,0.171429,0.161290,0,0
618,547,nda-16,2,Contradiction,0.710200,0.674000,0.004130,0.002443,0.285714,0.285714,0,0
300,35,nda-20,2,Contradiction,0.702511,0.511111,0.004118,0.002313,0.250000,0.250000,0,0
457,44,nda-7,2,Contradiction,0.686051,0.363649,0.004264,0.002674,0.178571,0.185185,0,0
656,563,nda-2,2,Contradiction,0.684250,0.566667,0.065000,0.047500,0.500000,0.500000,0,0
809,610,nda-2,2,Contradiction,0.682641,0.463333,0.004000,0.002293,0.210526,0.157895,0,0


In [42]:
detected_keys = detected[
    ["document_id", "label_id"]
]

detected_pairs = dev_pair_df.merge(
    detected_keys,
    on=["document_id", "label_id"],
    how="inner"
)

detected_pairs = (
    detected_pairs
    .sort_values(
        [
            "document_id",
            "label_id",
            "final_conflict_score"
        ],
        ascending=[True, True, False]
    )
)

display(
    detected_pairs[
        [
            "document_id",
            "label_id",
            "span_a",
            "span_b",
            "modality_a",
            "modality_b",
            "core_prop_similarity",
            "scope_compatibility",
            "strict_conflict_score",
            "final_conflict_score",
            "text_a",
            "text_b"
        ]
    ].head(20)
)

,document_id,label_id,span_a,span_b,modality_a,modality_b,core_prop_similarity,scope_compatibility,strict_conflict_score,final_conflict_score,text_a,text_b
6,10,nda-1,90,22,none,none,0.222222,1.0,0.500000,0.176275,"Additionally, nothing in this CNDA constitutes...",ARTICLE 2- CONFIDENTIAL INFORMATION
3,10,nda-1,44,22,obligation,none,0.062500,1.0,0.478431,0.168272,"Upon the Disclosing Party’s written request, t...",ARTICLE 2- CONFIDENTIAL INFORMATION
1,10,nda-1,44,30,obligation,none,0.088235,1.0,0.452222,0.159816,"Upon the Disclosing Party’s written request, t...",(d) information that the Disclosing Party auth...
4,10,nda-1,90,30,none,none,0.166667,1.0,0.360000,0.127592,"Additionally, nothing in this CNDA constitutes...",(d) information that the Disclosing Party auth...
7,10,nda-1,30,49,none,obligation,0.062500,1.0,0.342222,0.121227,(d) information that the Disclosing Party auth...,The Receiving Party acknowledges and agrees th...
0,10,nda-1,44,90,obligation,none,0.078947,1.0,0.075833,0.027842,"Upon the Disclosing Party’s written request, t...","Additionally, nothing in this CNDA constitutes..."
8,10,nda-1,30,22,none,none,0.166667,1.0,0.076389,0.027819,(d) information that the Disclosing Party auth...,ARTICLE 2- CONFIDENTIAL INFORMATION
5,10,nda-1,90,49,none,obligation,0.055556,1.0,0.060556,0.022417,"Additionally, nothing in this CNDA constitutes...",The Receiving Party acknowledges and agrees th...
2,10,nda-1,44,49,obligation,obligation,0.089286,1.0,0.049643,0.018862,"Upon the Disclosing Party’s written request, t...",The Receiving Party acknowledges and agrees th...
9,10,nda-1,49,22,obligation,none,0.000000,1.0,0.025000,0.009350,The Receiving Party acknowledges and agrees th...,ARTICLE 2- CONFIDENTIAL INFORMATION


In [43]:
def generate_conflict_explanation(row):

    reasons = []

    if row.get("modality_conflict", 0) == 1:
        reasons.append(
            f"modality conflict: "
            f"{row.get('modality_a')} vs "
            f"{row.get('modality_b')}"
        )

    if row.get("negation_conflict", 0) == 1:
        reasons.append(
            "opposing negation"
        )

    if row.get("scope_compatibility", 0) >= 0.8:
        reasons.append(
            "compatible legal scope"
        )

    if row.get("core_prop_similarity", 0) >= 0.4:
        reasons.append(
            "similar core proposition"
        )

    if row.get("proposition_jaccard", 0) >= 0.3:
        reasons.append(
            "substantial proposition overlap"
        )

    if row.get("condition_similarity", 0) >= 0.7:
        reasons.append(
            "similar conditions"
        )

    if not reasons:
        reasons.append(
            "semantic and structural conflict signals"
        )

    return "; ".join(reasons)


detected_pairs["explanation_basis"] = (
    detected_pairs.apply(
        generate_conflict_explanation,
        axis=1
    )
)

display(
    detected_pairs[
        [
            "document_id",
            "label_id",
            "span_a",
            "span_b",
            "modality_a",
            "modality_b",
            "explanation_basis",
            "text_a",
            "text_b"
        ]
    ].head(20)
)

,document_id,label_id,span_a,span_b,modality_a,modality_b,explanation_basis,text_a,text_b
6,10,nda-1,90,22,none,none,compatible legal scope; similar conditions,"Additionally, nothing in this CNDA constitutes...",ARTICLE 2- CONFIDENTIAL INFORMATION
3,10,nda-1,44,22,obligation,none,compatible legal scope; similar conditions,"Upon the Disclosing Party’s written request, t...",ARTICLE 2- CONFIDENTIAL INFORMATION
1,10,nda-1,44,30,obligation,none,compatible legal scope; similar conditions,"Upon the Disclosing Party’s written request, t...",(d) information that the Disclosing Party auth...
4,10,nda-1,90,30,none,none,compatible legal scope; similar conditions,"Additionally, nothing in this CNDA constitutes...",(d) information that the Disclosing Party auth...
7,10,nda-1,30,49,none,obligation,compatible legal scope; similar conditions,(d) information that the Disclosing Party auth...,The Receiving Party acknowledges and agrees th...
0,10,nda-1,44,90,obligation,none,compatible legal scope; similar conditions,"Upon the Disclosing Party’s written request, t...","Additionally, nothing in this CNDA constitutes..."
8,10,nda-1,30,22,none,none,compatible legal scope; similar conditions,(d) information that the Disclosing Party auth...,ARTICLE 2- CONFIDENTIAL INFORMATION
5,10,nda-1,90,49,none,obligation,compatible legal scope; similar conditions,"Additionally, nothing in this CNDA constitutes...",The Receiving Party acknowledges and agrees th...
2,10,nda-1,44,49,obligation,obligation,compatible legal scope; similar conditions,"Upon the Disclosing Party’s written request, t...",The Receiving Party acknowledges and agrees th...
9,10,nda-1,49,22,obligation,none,compatible legal scope; similar conditions,The Receiving Party acknowledges and agrees th...,ARTICLE 2- CONFIDENTIAL INFORMATION


In [44]:
def build_evidence_record(row):

    return {
        "document_id": row["document_id"],
        "label_id": row["label_id"],
        "span_a": row["span_a"],
        "span_b": row["span_b"],
        "clause_a": row["text_a"],
        "clause_b": row["text_b"],
        "modality_a": row["modality_a"],
        "modality_b": row["modality_b"],
        "conflict_score": row["final_conflict_score"],
        "explanation": row["explanation_basis"]
    }


evidence_records = (
    detected_pairs
    .apply(build_evidence_record, axis=1)
    .tolist()
)

print(
    "Evidence-backed contradiction records:",
    len(evidence_records)
)

if evidence_records:
    display(
        pd.DataFrame(evidence_records).head(10)
    )

Evidence-backed contradiction records: 1080


,document_id,label_id,span_a,span_b,clause_a,clause_b,modality_a,modality_b,conflict_score,explanation
0,10,nda-1,90,22,"Additionally, nothing in this CNDA constitutes...",ARTICLE 2- CONFIDENTIAL INFORMATION,none,none,0.176275,compatible legal scope; similar conditions
1,10,nda-1,44,22,"Upon the Disclosing Party’s written request, t...",ARTICLE 2- CONFIDENTIAL INFORMATION,obligation,none,0.168272,compatible legal scope; similar conditions
2,10,nda-1,44,30,"Upon the Disclosing Party’s written request, t...",(d) information that the Disclosing Party auth...,obligation,none,0.159816,compatible legal scope; similar conditions
3,10,nda-1,90,30,"Additionally, nothing in this CNDA constitutes...",(d) information that the Disclosing Party auth...,none,none,0.127592,compatible legal scope; similar conditions
4,10,nda-1,30,49,(d) information that the Disclosing Party auth...,The Receiving Party acknowledges and agrees th...,none,obligation,0.121227,compatible legal scope; similar conditions
5,10,nda-1,44,90,"Upon the Disclosing Party’s written request, t...","Additionally, nothing in this CNDA constitutes...",obligation,none,0.027842,compatible legal scope; similar conditions
6,10,nda-1,30,22,(d) information that the Disclosing Party auth...,ARTICLE 2- CONFIDENTIAL INFORMATION,none,none,0.027819,compatible legal scope; similar conditions
7,10,nda-1,90,49,"Additionally, nothing in this CNDA constitutes...",The Receiving Party acknowledges and agrees th...,none,obligation,0.022417,compatible legal scope; similar conditions
8,10,nda-1,44,49,"Upon the Disclosing Party’s written request, t...",The Receiving Party acknowledges and agrees th...,obligation,obligation,0.018862,compatible legal scope; similar conditions
9,10,nda-1,49,22,The Receiving Party acknowledges and agrees th...,ARTICLE 2- CONFIDENTIAL INFORMATION,obligation,none,0.009350,compatible legal scope; similar conditions


In [45]:
# ============================================================
# PART 1: GOLD CONTRADICTION PAIR EVALUATION
# ============================================================

import pandas as pd
import numpy as np

df = dev_pair_df.copy()

# ------------------------------------------------------------
# 1. Safely convert gold_spans to sets
# ------------------------------------------------------------

def parse_span_set(x):

    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    try:
        if pd.isna(x):
            return set()
    except:
        pass

    if isinstance(x, str):
        try:
            value = eval(x)

            if isinstance(value, (list, tuple, set)):
                return set(value)

        except:
            pass

    return set()


df["gold_span_set"] = df["gold_spans"].apply(parse_span_set)

# ------------------------------------------------------------
# 2. Check whether A/B are gold evidence
# ------------------------------------------------------------

df["a_is_gold"] = df.apply(
    lambda r: r["span_a"] in r["gold_span_set"],
    axis=1
)

df["b_is_gold"] = df.apply(
    lambda r: r["span_b"] in r["gold_span_set"],
    axis=1
)

df["pair_touches_gold"] = (
    df["a_is_gold"] |
    df["b_is_gold"]
)

df["pair_both_gold"] = (
    df["a_is_gold"] &
    df["b_is_gold"]
)

# ------------------------------------------------------------
# 3. Gold contradiction condition
#
# target:
# 0 = NotMentioned
# 1 = Entailment
# 2 = Contradiction
# ------------------------------------------------------------

df["gold_contradiction_example"] = (
    df["target"] == 2
)

# Candidate contradiction pair must:
# - belong to a contradiction example
# - touch gold evidence
#
# We initially use pair_touches_gold rather than requiring both
# spans to be gold, because a contradiction may be represented
# by one gold clause plus one retrieved comparison clause.

df["gold_contradiction_candidate"] = (
    df["gold_contradiction_example"] &
    df["pair_touches_gold"]
)

print("Total pairs:", len(df))

print(
    "Contradiction examples:",
    df["gold_contradiction_example"].sum()
)

print(
    "Gold-touching contradiction pairs:",
    df["gold_contradiction_candidate"].sum()
)

print(
    "Pairs with both spans gold:",
    df["pair_both_gold"].sum()
)

Total pairs: 10370
Contradiction examples: 950
Gold-touching contradiction pairs: 208
Pairs with both spans gold: 82


In [47]:
# ============================================================
# PART 3: CONFLICT THRESHOLD EVALUATION
# ============================================================

def evaluate_conflict_threshold(
    df,
    threshold,
    score_col="final_conflict_score"
):

    predicted = (
        df[score_col] >= threshold
    )

    actual = (
        df["gold_contradiction_candidate"]
    )

    tp = (predicted & actual).sum()
    fp = (predicted & ~actual).sum()
    fn = (~predicted & actual).sum()

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


thresholds = np.arange(
    0.10,
    0.91,
    0.05
)

results = []

for threshold in thresholds:

    results.append(
        evaluate_conflict_threshold(
            df,
            threshold
        )
    )


threshold_results = pd.DataFrame(results)

display(
    threshold_results.sort_values(
        "f1",
        ascending=False
    )
)

,threshold,TP,FP,FN,precision,recall,f1
0,0.10,98,4293,110,0.022318,0.471154,0.042618
1,0.15,64,2852,144,0.021948,0.307692,0.040973
2,0.20,27,1239,181,0.021327,0.129808,0.036635
3,0.25,11,527,197,0.020446,0.052885,0.029491
4,0.30,8,378,200,0.020725,0.038462,0.026936
5,0.35,4,155,204,0.025157,0.019231,0.021798
7,0.45,1,34,207,0.028571,0.004808,0.008230
6,0.40,1,46,207,0.021277,0.004808,0.007843
8,0.50,0,18,208,0.000000,0.000000,0.000000
9,0.55,0,13,208,0.000000,0.000000,0.000000


In [46]:
# ============================================================
# PART 2: SCORE DISTRIBUTION FOR GOLD CONTRADICTION PAIRS
# ============================================================

score_col = "final_conflict_score"

if score_col not in df.columns:

    print(
        f"{score_col} is missing."
    )

    print(
        "Available score columns:"
    )

    print([
        c for c in df.columns
        if "score" in c.lower()
    ])

else:

    gold_conflict = df[
        df["gold_contradiction_candidate"]
    ]

    non_gold = df[
        ~df["gold_contradiction_candidate"]
    ]

    print("\nGold contradiction pair score statistics:")
    print(
        gold_conflict[score_col].describe()
    )

    print("\nNon-gold pair score statistics:")
    print(
        non_gold[score_col].describe()
    )


Gold contradiction pair score statistics:
count    208.000000
mean       0.098927
std        0.096059
min        0.000646
25%        0.017194
50%        0.057508
75%        0.172523
max        0.475166
Name: final_conflict_score, dtype: float64

Non-gold pair score statistics:
count    10162.000000
mean         0.091163
std          0.096993
min          0.000000
25%          0.013124
50%          0.027186
75%          0.169452
max          0.883750
Name: final_conflict_score, dtype: float64


In [48]:
# ============================================================
# PART 4: BEST CONFLICT THRESHOLD
# ============================================================

best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

print(
    "Best threshold:",
    round(best_row["threshold"], 3)
)

print(
    "Precision:",
    round(best_row["precision"], 4)
)

print(
    "Recall:",
    round(best_row["recall"], 4)
)

print(
    "F1:",
    round(best_row["f1"], 4)
)

Best threshold: 0.1
Precision: 0.0223
Recall: 0.4712
F1: 0.0426


In [49]:
# ============================================================
# PART 5: TOP PREDICTED CONTRADICTIONS
# ============================================================

best_threshold = best_row["threshold"]

predicted_conflicts = df[
    df["final_conflict_score"] >= best_threshold
].copy()

predicted_conflicts = predicted_conflicts.sort_values(
    "final_conflict_score",
    ascending=False
)

print(
    "Predicted conflict pairs:",
    len(predicted_conflicts)
)

display(
    predicted_conflicts[
        [
            "document_id",
            "label_id",
            "span_a",
            "span_b",
            "target",
            "modality_a",
            "modality_b",
            "core_prop_similarity",
            "proposition_jaccard",
            "scope_compatibility",
            "strict_conflict_score",
            "final_conflict_score",
            "pair_touches_gold",
            "text_a",
            "text_b"
        ]
    ].head(30)
)

Predicted conflict pairs: 4391


,document_id,label_id,span_a,span_b,target,modality_a,modality_b,core_prop_similarity,proposition_jaccard,scope_compatibility,strict_conflict_score,final_conflict_score,pair_touches_gold,text_a,text_b
7147,590,nda-1,11,7,0,prohibition,obligation,1.000000,0.750000,1.0,0.900000,0.883750,False,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7260,590,nda-3,11,7,1,prohibition,obligation,1.000000,0.750000,1.0,0.900000,0.883750,False,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7240,590,nda-2,7,11,2,obligation,prohibition,1.000000,0.750000,1.0,0.900000,0.883750,False,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
6810,582,nda-10,13,21,1,prohibition,permission,0.430769,0.428571,1.0,0.702336,0.703390,True,"You and the Company also each agree, on behalf...",4. In the event you or any of your Representat...
361,13,nda-11,43,111,0,permission,prohibition,0.363636,0.363636,1.0,0.630952,0.652197,False,A Disclosing Party may disclose or discuss a P...,Disclosure of Confidential Information of any ...
4689,440,nda-19,19,7,0,prohibition,obligation,0.200000,0.400000,1.0,0.866667,0.632733,False,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
4647,440,nda-15,19,7,1,prohibition,obligation,0.200000,0.400000,1.0,0.866667,0.632733,False,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
9160,71,nda-7,10,9,1,obligation,prohibition,0.322581,0.321429,1.0,0.585606,0.622972,True,Prospect’s and Prospect’s Broker’s disseminati...,Prospect and Prospect’s Broker acknowledge tha...
9133,71,nda-3,9,10,0,prohibition,obligation,0.322581,0.321429,1.0,0.585606,0.622972,False,Prospect and Prospect’s Broker acknowledge tha...,Prospect’s and Prospect’s Broker’s disseminati...
9113,71,nda-2,9,10,0,prohibition,obligation,0.322581,0.321429,1.0,0.585606,0.622972,False,Prospect and Prospect’s Broker acknowledge tha...,Prospect’s and Prospect’s Broker’s disseminati...


In [50]:
# ============================================================
# PART 6: FINAL CONFLICT FEATURE TABLE
# ============================================================

feature_cols = [
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "aligned_conflict_score",
    "clause_reasoning_score_a",
    "clause_reasoning_score_b",
    "modality_conflict",
    "negation_conflict",
    "condition_similarity",
    "party_similarity"
]

available_features = [
    c for c in feature_cols
    if c in df.columns
]

print("Available conflict features:")
print(available_features)

display(
    df[available_features].describe().T
)

Available conflict features:
['core_prop_similarity', 'proposition_jaccard', 'scope_compatibility', 'strict_conflict_score', 'aligned_conflict_score', 'clause_reasoning_score_a', 'clause_reasoning_score_b', 'modality_conflict', 'negation_conflict', 'condition_similarity', 'party_similarity']


,count,mean,std,min,25%,50%,75%,max
core_prop_similarity,10370.0,0.105230,0.108287,0.0,0.035714,0.085106,0.142857,1.0000
proposition_jaccard,10370.0,0.106337,0.115252,0.0,0.037037,0.083333,0.142857,1.0000
scope_compatibility,10370.0,0.801254,0.284943,0.0,0.500000,1.000000,1.000000,1.0000
strict_conflict_score,10370.0,0.187179,0.211089,0.0,0.036075,0.065625,0.363649,0.9000
aligned_conflict_score,10370.0,0.008844,0.027857,0.0,0.002000,0.003000,0.004120,0.7625
clause_reasoning_score_a,10370.0,0.253804,0.131220,0.0,0.164997,0.241579,0.336433,0.9500
clause_reasoning_score_b,10370.0,0.253804,0.131220,0.0,0.164997,0.241579,0.336433,0.9500
modality_conflict,10370.0,0.148698,0.355808,0.0,0.000000,0.000000,0.000000,1.0000
negation_conflict,10370.0,0.135873,0.342670,0.0,0.000000,0.000000,0.000000,1.0000
condition_similarity,10370.0,0.645162,0.478465,0.0,0.000000,1.000000,1.000000,1.0000


In [51]:
# ============================================================
# NEXT STEP: LOAD SAVED PROJECT DATA
# ============================================================

import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/aditik1234/legal-data/legaldatawhatever"

clauses_df = pd.read_pickle(f"{BASE}/clauses_df.pkl")
structured_clauses = pd.read_pickle(f"{BASE}/structured_clauses.pkl")
ml_df = pd.read_pickle(f"{BASE}/ml_df.pkl")
dev_pair_df = pd.read_pickle(f"{BASE}/dev_pair_df.pkl")
dev_structured_df = pd.read_pickle(f"{BASE}/dev_structured_df.pkl")

print("clauses_df:", clauses_df.shape)
print("structured_clauses:", structured_clauses.shape)
print("ml_df:", ml_df.shape)
print("dev_pair_df:", dev_pair_df.shape)
print("dev_structured_df:", dev_structured_df.shape)

clauses_df: (47321, 3)
structured_clauses: (47321, 9)
ml_df: (10319, 10)
dev_pair_df: (10370, 48)
dev_structured_df: (5185, 14)


In [52]:
# ============================================================
# GOLD EVIDENCE LOOKUP
# ============================================================

def parse_span_set(x):
    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    if isinstance(x, str):
        try:
            value = eval(x)
            if isinstance(value, (list, tuple, set)):
                return set(value)
        except Exception:
            pass

    try:
        if pd.isna(x):
            return set()
    except Exception:
        pass

    return set()


dev_pairs = dev_pair_df.copy()

dev_pairs["gold_span_set"] = (
    dev_pairs["gold_spans"]
    .apply(parse_span_set)
)

dev_pairs["pair_touches_gold"] = (
    dev_pairs.apply(
        lambda r:
        (r["span_a"] in r["gold_span_set"]) or
        (r["span_b"] in r["gold_span_set"]),
        axis=1
    )
)

print(
    "Pairs touching gold evidence:",
    dev_pairs["pair_touches_gold"].sum()
)

print(
    "Total pairs:",
    len(dev_pairs)
)

Pairs touching gold evidence: 1838
Total pairs: 10370


In [53]:
# ============================================================
# HYBRID EVIDENCE RANKING SCORE
# ============================================================

df = dev_pairs.copy()

# Safely fill missing numerical values
feature_cols = [
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "aligned_conflict_score",
    "clause_reasoning_score_a",
    "clause_reasoning_score_b",
    "modality_conflict",
    "negation_conflict",
    "condition_similarity",
    "party_similarity"
]

for col in feature_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        ).fillna(0.0)

# Average reasoning quality
df["reasoning_mean"] = (
    df["clause_reasoning_score_a"] +
    df["clause_reasoning_score_b"]
) / 2

# ------------------------------------------------------------
# Hybrid score
# ------------------------------------------------------------

df["hybrid_evidence_score"] = (
    0.25 * df["core_prop_similarity"] +
    0.15 * df["proposition_jaccard"] +
    0.10 * df["scope_compatibility"] +
    0.15 * df["strict_conflict_score"] +
    0.15 * df["aligned_conflict_score"] +
    0.10 * df["modality_conflict"] +
    0.05 * df["negation_conflict"] +
    0.05 * df["reasoning_mean"]
)

print(
    df["hybrid_evidence_score"].describe()
)

count    10370.000000
mean         0.186141
std          0.097312
min          0.000000
25%          0.116395
50%          0.160796
75%          0.246463
max          0.909375
Name: hybrid_evidence_score, dtype: float64


In [54]:
# ============================================================
# EVIDENCE RANKING EVALUATION
# ============================================================

def evaluate_recall_at_k(df, score_col, k_values=[1, 3, 5, 10]):
    
    results = []
    
    for k in k_values:
        
        hits = 0
        total = 0
        
        for (doc_id, label_id), group in df.groupby(
            ["document_id", "label_id"]
        ):
            
            group = group.sort_values(
                score_col,
                ascending=False
            )
            
            top_k = group.head(k)
            
            gold = set()
            
            for x in group["gold_span_set"]:
                gold |= set(x)
            
            retrieved = set(
                top_k["span_a"].tolist() +
                top_k["span_b"].tolist()
            )
            
            if len(gold) > 0:
                total += 1
                
                if len(gold & retrieved) > 0:
                    hits += 1
        
        recall = hits / total if total else 0
        
        results.append({
            "K": k,
            "Hits": hits,
            "Total": total,
            "Recall": recall
        })
    
    return pd.DataFrame(results)


hybrid_results = evaluate_recall_at_k(
    df,
    "hybrid_evidence_score",
    [1, 3, 5, 10]
)

display(hybrid_results)

,K,Hits,Total,Recall
0,1,178,614,0.289902
1,3,317,614,0.516287
2,5,374,614,0.609121
3,10,404,614,0.657980


In [56]:
# ============================================================
# RECONSTRUCT FINAL CONFLICT SCORE
# ============================================================

import pandas as pd
import numpy as np

df = dev_pair_df.copy()

# ------------------------------------------------------------
# Required features
# ------------------------------------------------------------

required_features = [
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "aligned_conflict_score",
    "clause_reasoning_score_a",
    "clause_reasoning_score_b",
    "modality_conflict",
    "negation_conflict",
    "condition_similarity",
    "party_similarity"
]

available_features = [
    c for c in required_features
    if c in df.columns
]

print("Available conflict features:")
print(available_features)

# ------------------------------------------------------------
# Normalize helper
# ------------------------------------------------------------

def safe_num(series):
    return pd.to_numeric(
        series,
        errors="coerce"
    ).fillna(0.0)


# ------------------------------------------------------------
# Build components
# ------------------------------------------------------------

core = safe_num(
    df["core_prop_similarity"]
)

jaccard = safe_num(
    df["proposition_jaccard"]
)

scope = safe_num(
    df["scope_compatibility"]
)

strict = safe_num(
    df["strict_conflict_score"]
)

aligned = safe_num(
    df["aligned_conflict_score"]
)

reasoning = (
    safe_num(df["clause_reasoning_score_a"]) +
    safe_num(df["clause_reasoning_score_b"])
) / 2

modality = safe_num(
    df["modality_conflict"]
)

negation = safe_num(
    df["negation_conflict"]
)

condition = safe_num(
    df["condition_similarity"]
)

party = safe_num(
    df["party_similarity"]
)


# ------------------------------------------------------------
# Reconstruct conflict score
# ------------------------------------------------------------

df["final_conflict_score"] = (
    0.20 * core +
    0.10 * jaccard +
    0.10 * scope +
    0.25 * strict +
    0.10 * aligned +
    0.05 * reasoning +
    0.10 * modality +
    0.05 * negation +
    0.025 * condition +
    0.025 * party
)

print("\nFinal conflict score:")
print(
    df["final_conflict_score"].describe()
)

display(
    df[
        [
            "document_id",
            "label_id",
            "span_a",
            "span_b",
            "target",
            "final_conflict_score"
        ]
    ]
    .sort_values(
        "final_conflict_score",
        ascending=False
    )
    .head(20)
)

Available conflict features:
['core_prop_similarity', 'proposition_jaccard', 'scope_compatibility', 'strict_conflict_score', 'aligned_conflict_score', 'clause_reasoning_score_a', 'clause_reasoning_score_b', 'modality_conflict', 'negation_conflict', 'condition_similarity', 'party_similarity']

Final conflict score:
count    10370.000000
mean         0.223100
std          0.111980
min          0.000000
25%          0.145785
50%          0.197207
75%          0.302628
max          0.923750
Name: final_conflict_score, dtype: float64


,document_id,label_id,span_a,span_b,target,final_conflict_score
7240,590,nda-2,7,11,2,0.923750
7260,590,nda-3,11,7,1,0.923750
7147,590,nda-1,11,7,0,0.923750
9485,73,nda-4,133,37,1,0.717500
9500,73,nda-7,133,37,1,0.717500
9490,73,nda-5,133,37,1,0.717500
9390,73,nda-13,133,37,1,0.717500
9380,73,nda-12,133,37,1,0.717500
9420,73,nda-17,133,37,2,0.717500
9467,73,nda-20,133,37,2,0.717500


In [58]:
# ============================================================
# HYBRID EVIDENCE + CONFLICT SCORE
# ============================================================

# Make sure hybrid evidence score exists
if "hybrid_evidence_score" not in df.columns:

    # Reconstruct a retrieval/evidence component
    evidence_components = []

    for c in [
        "retrieval_a",
        "retrieval_b",
        "core_prop_similarity",
        "proposition_jaccard",
        "condition_similarity",
        "party_similarity"
    ]:
        if c in df.columns:
            evidence_components.append(
                safe_num(df[c])
            )

    if len(evidence_components) > 0:
        df["hybrid_evidence_score"] = sum(
            evidence_components
        ) / len(evidence_components)

    else:
        df["hybrid_evidence_score"] = 0.0


# ------------------------------------------------------------
# Combined novelty/conflict score
# ------------------------------------------------------------

df["novelty_conflict_score"] = (
    0.50 * df["hybrid_evidence_score"] +
    0.50 * df["final_conflict_score"]
)

print("Novelty/conflict score:")
print(
    df["novelty_conflict_score"].describe()
)

Novelty/conflict score:
count    10370.000000
mean         0.241623
std          0.108892
min          0.001901
25%          0.169172
50%          0.235160
75%          0.306249
max          0.822866
Name: novelty_conflict_score, dtype: float64


In [59]:
# ============================================================
# GOLD-AWARE NOVELTY DIAGNOSTIC
# ============================================================

def parse_span_set(x):

    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    if isinstance(x, str):
        try:
            value = eval(x)

            if isinstance(
                value,
                (list, tuple, set)
            ):
                return set(value)

        except:
            pass

    try:
        if pd.isna(x):
            return set()
    except:
        pass

    return set()


# ------------------------------------------------------------
# Gold spans
# ------------------------------------------------------------

df["gold_span_set"] = (
    df["gold_spans"]
    .apply(parse_span_set)
)


# ------------------------------------------------------------
# Does pair touch gold?
# ------------------------------------------------------------

df["pair_touches_gold"] = (
    df.apply(
        lambda r:
        (
            r["span_a"] in r["gold_span_set"]
            or
            r["span_b"] in r["gold_span_set"]
        ),
        axis=1
    )
)


# ------------------------------------------------------------
# Gold contradiction candidates
# ------------------------------------------------------------

# target == 1 = Contradiction
df["gold_contradiction_pair"] = (
    (df["target"] == 1) &
    (df["pair_touches_gold"])
)


print(
    "Gold-touching pairs:",
    df["pair_touches_gold"].sum()
)

print(
    "Gold contradiction pairs:",
    df["gold_contradiction_pair"].sum()
)

Gold-touching pairs: 1838
Gold contradiction pairs: 1630


In [60]:
# ============================================================
# NOVELTY-CONFLICT RANKING EVALUATION
# ============================================================

# Sort all candidate pairs by the combined score
ranked = df.sort_values(
    "novelty_conflict_score",
    ascending=False
).copy()


# ------------------------------------------------------------
# Top-K evaluation
# ------------------------------------------------------------

for k in [1, 3, 5, 10]:

    top_k = (
        ranked
        .groupby(
            ["document_id", "label_id"],
            sort=False
        )
        .head(k)
    )

    examples = (
        df[
            ["document_id", "label_id"]
        ]
        .drop_duplicates()
    )

    hits = (
        top_k
        .groupby(
            ["document_id", "label_id"]
        )["pair_touches_gold"]
        .max()
        .fillna(False)
    )

    hit_count = hits.sum()

    recall = (
        hit_count /
        len(examples)
    )

    print(
        f"Novelty Evidence Recall@{k}: "
        f"{recall:.4f} "
        f"({recall*100:.2f}%)"
    )

Novelty Evidence Recall@1: 0.1697 (16.97%)
Novelty Evidence Recall@3: 0.2922 (29.22%)
Novelty Evidence Recall@5: 0.3529 (35.29%)
Novelty Evidence Recall@10: 0.3896 (38.96%)


In [61]:
# ============================================================
# CONTRADICTION THRESHOLD SEARCH
# ============================================================

evaluation_df = df.copy()

evaluation_df["predicted_conflict"] = False

threshold_results = []

for threshold in np.arange(
    0.05,
    0.91,
    0.05
):

    pred = (
        evaluation_df["novelty_conflict_score"]
        >= threshold
    )

    # Gold contradiction definition
    gold = (
        evaluation_df["gold_contradiction_pair"]
    )

    tp = (
        pred & gold
    ).sum()

    fp = (
        pred & ~gold
    ).sum()

    fn = (
        ~pred & gold
    ).sum()

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    threshold_results.append(
        [
            threshold,
            precision,
            recall,
            f1,
            tp,
            fp,
            fn
        ]
    )


threshold_df = pd.DataFrame(
    threshold_results,
    columns=[
        "threshold",
        "precision",
        "recall",
        "f1",
        "TP",
        "FP",
        "FN"
    ]
)

display(
    threshold_df.sort_values(
        "f1",
        ascending=False
    ).head(10)
)

,threshold,precision,recall,f1,TP,FP,FN
0,0.05,0.158303,0.975460,0.272400,1590,8454,40
1,0.10,0.156790,0.907362,0.267378,1479,7954,151
2,0.15,0.155245,0.817178,0.260921,1332,7248,298
3,0.20,0.154487,0.603067,0.245965,983,5380,647
4,0.25,0.152707,0.442945,0.227115,722,4006,908
5,0.30,0.160953,0.273620,0.202681,446,2325,1184
6,0.35,0.156073,0.152147,0.154085,248,1341,1382
7,0.40,0.137333,0.063190,0.086555,103,647,1527
8,0.45,0.111111,0.025153,0.041021,41,328,1589
9,0.50,0.105528,0.012883,0.022963,21,178,1609


In [62]:
# ============================================================
# CLAIM-LEVEL NOVELTY EVIDENCE
# ============================================================

claim_scores = (
    df.groupby(
        ["document_id", "label_id", "target"],
        as_index=False
    )
    .agg(
        max_novelty_conflict_score=(
            "novelty_conflict_score",
            "max"
        ),
        mean_novelty_conflict_score=(
            "novelty_conflict_score",
            "mean"
        ),
        max_conflict_score=(
            "final_conflict_score",
            "max"
        ),
        max_evidence_score=(
            "hybrid_evidence_score",
            "max"
        ),
        pair_count=(
            "span_a",
            "count"
        ),
        gold_pair_count=(
            "gold_contradiction_pair",
            "sum"
        )
    )
)

print(
    "Claim-level examples:",
    len(claim_scores)
)

display(
    claim_scores
    .sort_values(
        "max_novelty_conflict_score",
        ascending=False
    )
    .head(20)
)

Claim-level examples: 1037


,document_id,label_id,target,max_novelty_conflict_score,mean_novelty_conflict_score,max_conflict_score,max_evidence_score,pair_count,gold_pair_count
724,590,nda-2,2,0.822866,0.324402,0.923750,0.721981,10,0
726,590,nda-3,1,0.798378,0.309916,0.923750,0.673006,10,0
714,590,nda-1,0,0.796290,0.338610,0.923750,0.668831,10,0
949,73,nda-5,1,0.743005,0.460897,0.717500,0.768510,10,4
939,73,nda-13,1,0.734898,0.352158,0.717500,0.752296,10,0
951,73,nda-8,1,0.734619,0.347427,0.717500,0.751739,10,4
943,73,nda-18,0,0.734234,0.355866,0.717500,0.750968,10,0
940,73,nda-15,1,0.733692,0.396528,0.717500,0.749883,10,4
938,73,nda-12,1,0.730440,0.386156,0.717500,0.743379,10,0
936,73,nda-10,1,0.726912,0.466350,0.717500,0.736323,10,7


In [63]:
# ============================================================
# NEXT STEP: FINAL NOVELTY EVIDENCE SCORE
# ============================================================

import pandas as pd
import numpy as np

df = dev_pair_df.copy()

# ------------------------------------------------------------
# 1. Verify available features
# ------------------------------------------------------------

features = [
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "aligned_conflict_score",
    "clause_reasoning_score_a",
    "clause_reasoning_score_b",
    "modality_conflict",
    "negation_conflict",
    "condition_similarity",
    "party_similarity"
]

available = [c for c in features if c in df.columns]

print("Available features:")
print(available)

# ------------------------------------------------------------
# 2. Normalize feature columns
# ------------------------------------------------------------

for c in available:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# ------------------------------------------------------------
# 3. Build evidence similarity score
# ------------------------------------------------------------

df["evidence_similarity_score"] = (
    0.35 * df["core_prop_similarity"] +
    0.20 * df["proposition_jaccard"] +
    0.15 * df["scope_compatibility"] +
    0.15 * df["condition_similarity"] +
    0.15 * df["party_similarity"]
)

# ------------------------------------------------------------
# 4. Build conflict score
# ------------------------------------------------------------

df["conflict_signal"] = (
    0.35 * df["strict_conflict_score"] +
    0.25 * df["aligned_conflict_score"] +
    0.20 * df["modality_conflict"] +
    0.10 * df["negation_conflict"] +
    0.10 * (
        (df["clause_reasoning_score_a"] +
         df["clause_reasoning_score_b"]) / 2
    )
)

# ------------------------------------------------------------
# 5. Combine evidence + conflict
# ------------------------------------------------------------

df["novelty_evidence_score"] = (
    0.60 * df["evidence_similarity_score"] +
    0.40 * df["conflict_signal"]
)

print("\nNovelty evidence score:")
print(
    df["novelty_evidence_score"].describe()
)

# ------------------------------------------------------------
# 6. Inspect highest scoring pairs
# ------------------------------------------------------------

inspect_cols = [
    "document_id",
    "label_id",
    "span_a",
    "span_b",
    "target",
    "evidence_similarity_score",
    "conflict_signal",
    "novelty_evidence_score",
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "aligned_conflict_score",
    "modality_conflict",
    "negation_conflict",
    "text_a",
    "text_b"
]

inspect_cols = [
    c for c in inspect_cols
    if c in df.columns
]

display(
    df.sort_values(
        "novelty_evidence_score",
        ascending=False
    )[inspect_cols].head(30)
)

Available features:
['core_prop_similarity', 'proposition_jaccard', 'scope_compatibility', 'strict_conflict_score', 'aligned_conflict_score', 'clause_reasoning_score_a', 'clause_reasoning_score_b', 'modality_conflict', 'negation_conflict', 'condition_similarity', 'party_similarity']

Novelty evidence score:
count    10370.000000
mean         0.266889
std          0.118725
min          0.000000
25%          0.189700
50%          0.262131
75%          0.336573
max          0.930250
Name: novelty_evidence_score, dtype: float64


,document_id,label_id,span_a,span_b,target,evidence_similarity_score,conflict_signal,novelty_evidence_score,core_prop_similarity,proposition_jaccard,scope_compatibility,strict_conflict_score,aligned_conflict_score,modality_conflict,negation_conflict,text_a,text_b
7147,590,nda-1,11,7,0,0.950000,0.900625,0.930250,1.000000,0.750000,1.0,0.900000,0.762500,1,1,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7240,590,nda-2,7,11,2,0.950000,0.900625,0.930250,1.000000,0.750000,1.0,0.900000,0.762500,1,1,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7260,590,nda-3,11,7,1,0.950000,0.900625,0.930250,1.000000,0.750000,1.0,0.900000,0.762500,1,1,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
9361,73,nda-10,133,37,1,1.000000,0.412500,0.765000,1.000000,1.000000,1.0,0.850000,0.100000,0,0,“RECEIVING PARTY”,The Receiving Party:
9380,73,nda-12,133,37,1,1.000000,0.412500,0.765000,1.000000,1.000000,1.0,0.850000,0.100000,0,0,“RECEIVING PARTY”,The Receiving Party:
9374,73,nda-11,133,37,0,1.000000,0.412500,0.765000,1.000000,1.000000,1.0,0.850000,0.100000,0,0,“RECEIVING PARTY”,The Receiving Party:
9510,73,nda-8,133,37,1,1.000000,0.412500,0.765000,1.000000,1.000000,1.0,0.850000,0.100000,0,0,“RECEIVING PARTY”,The Receiving Party:
9490,73,nda-5,133,37,1,1.000000,0.412500,0.765000,1.000000,1.000000,1.0,0.850000,0.100000,0,0,“RECEIVING PARTY”,The Receiving Party:
9500,73,nda-7,133,37,1,1.000000,0.412500,0.765000,1.000000,1.000000,1.0,0.850000,0.100000,0,0,“RECEIVING PARTY”,The Receiving Party:
9467,73,nda-20,133,37,2,1.000000,0.412500,0.765000,1.000000,1.000000,1.0,0.850000,0.100000,0,0,“RECEIVING PARTY”,The Receiving Party:


In [70]:
cols = [
    "document_id",
    "label_id",
    "span_a",
    "span_b",
    "target",
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "aligned_conflict_score",
    "modality_conflict",
    "negation_conflict",
    "pair_touches_gold",
    "text_a",
    "text_b"
]

In [72]:
# ============================================================
# NOVELTY EVIDENCE RECALL@K — FIXED COMPLETE BLOCK
# ============================================================

import pandas as pd
import numpy as np

df = dev_pair_df.copy()

# ------------------------------------------------------------
# 1. Parse gold spans
# ------------------------------------------------------------

def parse_span_set(x):

    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    if isinstance(x, str):
        try:
            value = eval(x)
            if isinstance(value, (list, tuple, set)):
                return set(value)
        except:
            pass

    return set()


df["gold_span_set"] = df["gold_spans"].apply(parse_span_set)

# ------------------------------------------------------------
# 2. Identify whether pair touches gold evidence
# ------------------------------------------------------------

df["pair_touches_gold"] = df.apply(
    lambda r:
        (r["span_a"] in r["gold_span_set"]) or
        (r["span_b"] in r["gold_span_set"]),
    axis=1
)

# ------------------------------------------------------------
# 3. CREATE NOVELTY EVIDENCE SCORE
# ------------------------------------------------------------
# Your dataframe currently has:
# core_prop_similarity
# proposition_jaccard
# scope_compatibility
# strict_conflict_score
# aligned_conflict_score
# modality_conflict
# negation_conflict
#
# We construct the novelty/evidence score from these available
# columns instead of trying to access a missing column.

# Evidence similarity component
evidence_similarity = (
    0.50 * df["core_prop_similarity"].fillna(0) +
    0.50 * df["proposition_jaccard"].fillna(0)
)

# Conflict component
conflict_signal = (
    0.70 * df["strict_conflict_score"].fillna(0) +
    0.30 * df["aligned_conflict_score"].fillna(0)
)

# Novelty evidence score
df["novelty_evidence_score"] = (
    0.60 * evidence_similarity +
    0.40 * conflict_signal
)

# ------------------------------------------------------------
# 4. Rank pairs within each example
# ------------------------------------------------------------

df = df.sort_values(
    ["document_id", "label_id", "novelty_evidence_score"],
    ascending=[True, True, False]
)

# ------------------------------------------------------------
# 5. Recall@K
# ------------------------------------------------------------

def recall_at_k(data, k):

    hits = 0
    total = 0

    for (doc, label), group in data.groupby(
        ["document_id", "label_id"]
    ):

        gold = set()

        for x in group["gold_span_set"]:
            gold.update(x)

        if len(gold) == 0:
            continue

        top_k = group.head(k)

        retrieved = (
            set(top_k["span_a"]) |
            set(top_k["span_b"])
        )

        if len(gold & retrieved) > 0:
            hits += 1

        total += 1

    return hits / total if total else 0


# ------------------------------------------------------------
# 6. Print Recall@K
# ------------------------------------------------------------

for k in [1, 3, 5, 10]:

    score = recall_at_k(df, k)

    print(
        f"Novelty Evidence Recall@{k}: "
        f"{score:.4f} ({score*100:.2f}%)"
    )

Novelty Evidence Recall@1: 0.2752 (27.52%)
Novelty Evidence Recall@3: 0.5228 (52.28%)
Novelty Evidence Recall@5: 0.6156 (61.56%)
Novelty Evidence Recall@10: 0.6580 (65.80%)


In [68]:
# ============================================================
# NEXT STEP: CONFLICT SCORE DIAGNOSTIC
# ============================================================

import pandas as pd
import numpy as np

df = dev_pair_df.copy()

# ------------------------------------------------------------
# 1. Check available columns
# ------------------------------------------------------------

print("Available columns:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 2. Reconstruct final conflict score
# ------------------------------------------------------------

# Use the strongest conflict-related signals already present.
# Keep this separate from evidence/novelty scoring.

df["final_conflict_score"] = (
    0.35 * df["strict_conflict_score"].fillna(0) +
    0.25 * df["aligned_conflict_score"].fillna(0) +
    0.20 * df["modality_conflict"].fillna(0) +
    0.15 * df["negation_conflict"].fillna(0) +
    0.05 * df["scope_compatibility"].fillna(0)
)

print("\nFinal conflict score:")
print(df["final_conflict_score"].describe())

# ------------------------------------------------------------
# 3. Gold span sets
# ------------------------------------------------------------

def parse_span_set(x):

    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    try:
        if pd.isna(x):
            return set()
    except:
        pass

    if isinstance(x, str):
        try:
            value = eval(x)

            if isinstance(value, (list, tuple, set)):
                return set(value)

        except:
            pass

    return set()


df["gold_span_set"] = df["gold_spans"].apply(parse_span_set)

# ------------------------------------------------------------
# 4. Determine whether pair touches gold evidence
# ------------------------------------------------------------

df["a_is_gold"] = df.apply(
    lambda r: r["span_a"] in r["gold_span_set"],
    axis=1
)

df["b_is_gold"] = df.apply(
    lambda r: r["span_b"] in r["gold_span_set"],
    axis=1
)

df["pair_touches_gold"] = (
    df["a_is_gold"] |
    df["b_is_gold"]
)

# ------------------------------------------------------------
# 5. Basic counts
# ------------------------------------------------------------

print("\nGold-touching pairs:", df["pair_touches_gold"].sum())
print("Non-gold pairs:", (~df["pair_touches_gold"]).sum())

# ------------------------------------------------------------
# 6. Score comparison
# ------------------------------------------------------------

print("\nFINAL CONFLICT SCORE BY GOLD CONTACT:")

display(
    df.groupby("pair_touches_gold")["final_conflict_score"]
      .agg(["count", "mean", "median", "max"])
)

# ------------------------------------------------------------
# 7. Top conflict candidates
# ------------------------------------------------------------

display(
    df.sort_values(
        "final_conflict_score",
        ascending=False
    )[
        [
            "document_id",
            "label_id",
            "span_a",
            "span_b",
            "target",
            "evidence_similarity_score",
            "conflict_signal",
            "novelty_evidence_score",
            "core_prop_similarity",
            "proposition_jaccard",
            "scope_compatibility",
            "strict_conflict_score",
            "aligned_conflict_score",
            "modality_conflict",
            "negation_conflict",
            "final_conflict_score",
            "pair_touches_gold",
            "text_a",
            "text_b"
        ]
    ].head(30)
)

# ------------------------------------------------------------
# 8. Top GOLD-touching conflict candidates
# ------------------------------------------------------------

gold_df = df[df["pair_touches_gold"]].copy()

print("\nTOP GOLD-TOUCHING PAIRS:")

display(
    gold_df.sort_values(
        "final_conflict_score",
        ascending=False
    )[
        [
            "document_id",
            "label_id",
            "span_a",
            "span_b",
            "target",
            "strict_conflict_score",
            "aligned_conflict_score",
            "modality_conflict",
            "negation_conflict",
            "final_conflict_score",
            "text_a",
            "text_b"
        ]
    ].head(20)
)

Available columns:
['document_id', 'label_id', 'target', 'span_a', 'span_b', 'text_a', 'text_b', 'retrieval_a', 'retrieval_b', 'modality_a', 'modality_b', 'modality_conflict', 'negation_conflict', 'proposition_similarity', 'condition_similarity', 'party_similarity', 'structured_conflict_score', 'proposition_overlap', 'strict_conflict_score', 'proposition_jaccard', 'aligned_conflict', 'aligned_conflict_score', 'subject_a', 'action_a', 'object_a', 'subject_b', 'action_b', 'object_b', 'proposition_alignment', 'structured_conflict_candidate', 'normalized_prop_a', 'normalized_prop_b', 'core_prop_similarity', 'conditions_a', 'conditions_b', 'temporal_a', 'temporal_b', 'numbers_a', 'numbers_b', 'scope_compatibility', 'exception_a', 'exception_b', 'scope_aware_score', 'conflict_category', 'gold_spans', 'evidence_hit', 'clause_reasoning_score_a', 'clause_reasoning_score_b']

Final conflict score:
count    10370.000000
mean         0.157907
std          0.144842
min          0.000000
25%        

,count,mean,median,max
pair_touches_gold,,,,
False,8532,0.157670,0.076636,0.905625
True,1838,0.159006,0.077414,0.797201


KeyError: "['evidence_similarity_score', 'conflict_signal', 'novelty_evidence_score'] not in index"

In [66]:
# ============================================================
# DIAGNOSTIC: FIND WHERE THE CURRENT SCORES ARE
# ============================================================

print("dev_pair_df columns:")
print(dev_pair_df.columns.tolist())

print("\n" + "="*70)

print("dev_structured_df columns:")
print(dev_structured_df.columns.tolist())

print("\n" + "="*70)

print("ml_df columns:")
print(ml_df.columns.tolist())

print("\n" + "="*70)

# Check which expected score columns exist in each dataframe

score_cols = [
    "evidence_similarity_score",
    "conflict_signal",
    "novelty_evidence_score",
    "hybrid_evidence_score",
    "final_conflict_score",
    "strict_conflict_score",
    "aligned_conflict_score"
]

for name, d in [
    ("dev_pair_df", dev_pair_df),
    ("dev_structured_df", dev_structured_df),
    ("ml_df", ml_df)
]:
    print(f"\n{name}:")
    print({
        c: c in d.columns
        for c in score_cols
    })

dev_pair_df columns:
['document_id', 'label_id', 'target', 'span_a', 'span_b', 'text_a', 'text_b', 'retrieval_a', 'retrieval_b', 'modality_a', 'modality_b', 'modality_conflict', 'negation_conflict', 'proposition_similarity', 'condition_similarity', 'party_similarity', 'structured_conflict_score', 'proposition_overlap', 'strict_conflict_score', 'proposition_jaccard', 'aligned_conflict', 'aligned_conflict_score', 'subject_a', 'action_a', 'object_a', 'subject_b', 'action_b', 'object_b', 'proposition_alignment', 'structured_conflict_candidate', 'normalized_prop_a', 'normalized_prop_b', 'core_prop_similarity', 'conditions_a', 'conditions_b', 'temporal_a', 'temporal_b', 'numbers_a', 'numbers_b', 'scope_compatibility', 'exception_a', 'exception_b', 'scope_aware_score', 'conflict_category', 'gold_spans', 'evidence_hit', 'clause_reasoning_score_a', 'clause_reasoning_score_b']

dev_structured_df columns:
['document_id', 'label_id', 'hypothesis', 'target', 'span_id', 'clause', 'retrieval_score', 

In [69]:
# ============================================================
# GOLD-AWARE CONFLICT SCORE DIAGNOSTIC
# ============================================================

import pandas as pd
import numpy as np

df = dev_pair_df.copy()

# ------------------------------------------------------------
# 1. Check columns
# ------------------------------------------------------------

print("Available columns:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 2. Identify whether each pair touches gold evidence
# ------------------------------------------------------------

def parse_span_set(x):

    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    try:
        if pd.isna(x):
            return set()
    except:
        pass

    if isinstance(x, str):
        try:
            value = eval(x)

            if isinstance(value, (list, tuple, set)):
                return set(value)

        except:
            pass

    return set()


df["gold_span_set"] = df["gold_spans"].apply(parse_span_set)

df["a_is_gold"] = df.apply(
    lambda r: r["span_a"] in r["gold_span_set"],
    axis=1
)

df["b_is_gold"] = df.apply(
    lambda r: r["span_b"] in r["gold_span_set"],
    axis=1
)

df["pair_touches_gold"] = (
    df["a_is_gold"] | df["b_is_gold"]
)

# ------------------------------------------------------------
# 3. Target distribution
# ------------------------------------------------------------

if "target" in df.columns:

    print("\nTarget distribution:")
    print(df["target"].value_counts(dropna=False))

# ------------------------------------------------------------
# 4. Available columns for inspection
# ------------------------------------------------------------

cols = [
    "document_id",
    "label_id",
    "span_a",
    "span_b",
    "target",
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "aligned_conflict_score",
    "modality_conflict",
    "negation_conflict",
    "pair_touches_gold",
    "text_a",
    "text_b"
]

cols = [c for c in cols if c in df.columns]

# ------------------------------------------------------------
# 5. Inspect highest conflict scores
# ------------------------------------------------------------

print("\nTOP CONFLICT PAIRS:")

display(
    df.sort_values(
        "strict_conflict_score",
        ascending=False
    )[cols].head(30)
)

# ------------------------------------------------------------
# 6. Compare gold vs non-gold pairs
# ------------------------------------------------------------

print("\nSTRICT CONFLICT SCORE BY GOLD CONTACT:")

print(
    df.groupby("pair_touches_gold")["strict_conflict_score"]
      .agg(["count", "mean", "max"])
)

# ------------------------------------------------------------
# 7. Gold-touching pairs
# ------------------------------------------------------------

gold_pairs = df[
    df["pair_touches_gold"]
].copy()

non_gold_pairs = df[
    ~df["pair_touches_gold"]
].copy()

print("\nGold-touching pairs:", len(gold_pairs))
print("Non-gold pairs:", len(non_gold_pairs))

# ------------------------------------------------------------
# 8. Statistics
# ------------------------------------------------------------

print("\nGold-touching conflict score statistics:")

print(
    gold_pairs["strict_conflict_score"].describe()
)

print("\nNon-gold conflict score statistics:")

print(
    non_gold_pairs["strict_conflict_score"].describe()
)

# ------------------------------------------------------------
# 9. Top gold-touching pairs
# ------------------------------------------------------------

print("\nTOP GOLD-TOUCHING CONFLICT PAIRS:")

display(
    gold_pairs.sort_values(
        "strict_conflict_score",
        ascending=False
    )[cols].head(20)
)

Available columns:
['document_id', 'label_id', 'target', 'span_a', 'span_b', 'text_a', 'text_b', 'retrieval_a', 'retrieval_b', 'modality_a', 'modality_b', 'modality_conflict', 'negation_conflict', 'proposition_similarity', 'condition_similarity', 'party_similarity', 'structured_conflict_score', 'proposition_overlap', 'strict_conflict_score', 'proposition_jaccard', 'aligned_conflict', 'aligned_conflict_score', 'subject_a', 'action_a', 'object_a', 'subject_b', 'action_b', 'object_b', 'proposition_alignment', 'structured_conflict_candidate', 'normalized_prop_a', 'normalized_prop_b', 'core_prop_similarity', 'conditions_a', 'conditions_b', 'temporal_a', 'temporal_b', 'numbers_a', 'numbers_b', 'scope_compatibility', 'exception_a', 'exception_b', 'scope_aware_score', 'conflict_category', 'gold_spans', 'evidence_hit', 'clause_reasoning_score_a', 'clause_reasoning_score_b']

Target distribution:
target
1    5190
0    4230
2     950
Name: count, dtype: int64

TOP CONFLICT PAIRS:


,document_id,label_id,span_a,span_b,target,core_prop_similarity,proposition_jaccard,scope_compatibility,strict_conflict_score,aligned_conflict_score,modality_conflict,negation_conflict,pair_touches_gold,text_a,text_b
7240,590,nda-2,7,11,2,1.000000,0.750000,1.0,0.900000,0.762500,1,1,False,"In addition, Proprietary Information shall inc...",Proprietary Information shall not include info...
7147,590,nda-1,11,7,0,1.000000,0.750000,1.0,0.900000,0.762500,1,1,False,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
7260,590,nda-3,11,7,1,1.000000,0.750000,1.0,0.900000,0.762500,1,1,False,Proprietary Information shall not include info...,"In addition, Proprietary Information shall inc..."
4647,440,nda-15,19,7,1,0.200000,0.400000,1.0,0.866667,0.563333,1,1,False,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
4689,440,nda-19,19,7,0,0.200000,0.400000,1.0,0.866667,0.563333,1,1,False,4. The obligations under this Agreement shall ...,1. This Agreement shall apply to:
9814,79,nda-3,10,6,0,0.250000,0.272727,1.0,0.857143,0.049416,1,1,False,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
9790,79,nda-2,10,6,2,0.250000,0.272727,1.0,0.857143,0.049416,1,1,True,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
9695,79,nda-1,10,6,0,0.250000,0.272727,1.0,0.857143,0.049416,1,1,False,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
9490,73,nda-5,133,37,1,1.000000,1.000000,1.0,0.850000,0.100000,0,0,True,“RECEIVING PARTY”,The Receiving Party:
9420,73,nda-17,133,37,2,1.000000,1.000000,1.0,0.850000,0.100000,0,0,True,“RECEIVING PARTY”,The Receiving Party:



STRICT CONFLICT SCORE BY GOLD CONTACT:
                   count      mean       max
pair_touches_gold                           
False               8532  0.187288  0.900000
True                1838  0.186671  0.857143

Gold-touching pairs: 1838
Non-gold pairs: 8532

Gold-touching conflict score statistics:
count    1838.000000
mean        0.186671
std         0.204366
min         0.000000
25%         0.041186
50%         0.069624
75%         0.352187
max         0.857143
Name: strict_conflict_score, dtype: float64

Non-gold conflict score statistics:
count    8532.000000
mean        0.187288
std         0.212521
min         0.000000
25%         0.034643
50%         0.064845
75%         0.366667
max         0.900000
Name: strict_conflict_score, dtype: float64

TOP GOLD-TOUCHING CONFLICT PAIRS:


,document_id,label_id,span_a,span_b,target,core_prop_similarity,proposition_jaccard,scope_compatibility,strict_conflict_score,aligned_conflict_score,modality_conflict,negation_conflict,pair_touches_gold,text_a,text_b
9790,79,nda-2,10,6,2,0.250000,0.272727,1.0,0.857143,0.049416,1,1,True,Confidential Information shall not include inf...,"The term ""Confidential Information"" shall mean..."
9490,73,nda-5,133,37,1,1.000000,1.000000,1.0,0.850000,0.100000,0,0,True,“RECEIVING PARTY”,The Receiving Party:
9361,73,nda-10,133,37,1,1.000000,1.000000,1.0,0.850000,0.100000,0,0,True,“RECEIVING PARTY”,The Receiving Party:
9420,73,nda-17,133,37,2,1.000000,1.000000,1.0,0.850000,0.100000,0,0,True,“RECEIVING PARTY”,The Receiving Party:
9485,73,nda-4,133,37,1,1.000000,1.000000,1.0,0.850000,0.100000,0,0,True,“RECEIVING PARTY”,The Receiving Party:
9500,73,nda-7,133,37,1,1.000000,1.000000,1.0,0.850000,0.100000,0,0,True,“RECEIVING PARTY”,The Receiving Party:
5790,507,nda-10,16,17,1,0.962963,0.961538,1.0,0.842857,0.097376,0,0,True,"Without the prior written consent of Big Sky, ...","Likewise, without the prior written consent of..."
7954,61,nda-4,25,23,1,0.045455,0.055556,1.0,0.817391,0.034674,1,1,True,3. Company shall not,Company also agrees that it shall treat the Co...
9487,73,nda-4,175,37,1,0.500000,1.000000,1.0,0.783333,0.091667,0,0,True,FOR THE RECEIVING PARTY:,The Receiving Party:
9426,73,nda-17,37,175,2,0.500000,1.000000,1.0,0.783333,0.091667,0,0,True,The Receiving Party:,FOR THE RECEIVING PARTY:


In [73]:
# ============================================================
# 1. VERIFY CURRENT DATAFRAME
# ============================================================

print("dev_pair_df shape:", dev_pair_df.shape)

print("\nColumns:")
print(dev_pair_df.columns.tolist())

required_cols = [
    "document_id",
    "label_id",
    "span_a",
    "span_b",
    "target",
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "aligned_conflict_score",
    "modality_conflict",
    "negation_conflict",
    "text_a",
    "text_b"
]

print("\nMissing required columns:")
print([c for c in required_cols if c not in dev_pair_df.columns])

dev_pair_df shape: (10370, 48)

Columns:
['document_id', 'label_id', 'target', 'span_a', 'span_b', 'text_a', 'text_b', 'retrieval_a', 'retrieval_b', 'modality_a', 'modality_b', 'modality_conflict', 'negation_conflict', 'proposition_similarity', 'condition_similarity', 'party_similarity', 'structured_conflict_score', 'proposition_overlap', 'strict_conflict_score', 'proposition_jaccard', 'aligned_conflict', 'aligned_conflict_score', 'subject_a', 'action_a', 'object_a', 'subject_b', 'action_b', 'object_b', 'proposition_alignment', 'structured_conflict_candidate', 'normalized_prop_a', 'normalized_prop_b', 'core_prop_similarity', 'conditions_a', 'conditions_b', 'temporal_a', 'temporal_b', 'numbers_a', 'numbers_b', 'scope_compatibility', 'exception_a', 'exception_b', 'scope_aware_score', 'conflict_category', 'gold_spans', 'evidence_hit', 'clause_reasoning_score_a', 'clause_reasoning_score_b']

Missing required columns:
[]


In [74]:
# ============================================================
# 2. BUILD NOVELTY EVIDENCE SCORE
# ============================================================

df = dev_pair_df.copy()

# Evidence similarity
evidence_similarity = (
    0.5 * df["core_prop_similarity"] +
    0.5 * df["proposition_jaccard"]
)

# Conflict signal
conflict_signal = (
    0.5 * df["strict_conflict_score"] +
    0.5 * df["aligned_conflict_score"]
)

# Novelty evidence score
df["novelty_evidence_score"] = (
    0.6 * evidence_similarity +
    0.4 * conflict_signal
)

print(df["novelty_evidence_score"].describe())

count    10370.000000
mean         0.102675
std          0.101499
min          0.000000
25%          0.032385
50%          0.073232
75%          0.151012
max          0.857500
Name: novelty_evidence_score, dtype: float64


In [75]:
# ============================================================
# 3. NOVELTY SCORE: GOLD VS NON-GOLD
# ============================================================

def parse_span_set(x):

    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    if isinstance(x, str):
        try:
            value = eval(x)
            if isinstance(value, (list, tuple, set)):
                return set(value)
        except:
            pass

    return set()


df["gold_span_set"] = df["gold_spans"].apply(parse_span_set)

df["pair_touches_gold"] = df.apply(
    lambda r:
    (r["span_a"] in r["gold_span_set"]) or
    (r["span_b"] in r["gold_span_set"]),
    axis=1
)

print("\nGold-touching pairs:")
print(df["pair_touches_gold"].value_counts())

print("\nNovelty score by gold contact:")
print(
    df.groupby("pair_touches_gold")["novelty_evidence_score"]
      .agg(["count", "mean", "median", "max"])
)


Gold-touching pairs:
pair_touches_gold
False    8532
True     1838
Name: count, dtype: int64

Novelty score by gold contact:
                   count      mean    median     max
pair_touches_gold                                   
False               8532  0.102708  0.071985  0.8575
True                1838  0.102521  0.077675  0.7900


In [76]:
# ============================================================
# 4. ROC-AUC + PR-AUC
# ============================================================

from sklearn.metrics import roc_auc_score, average_precision_score

y_true = df["pair_touches_gold"].astype(int)
y_score = df["novelty_evidence_score"].fillna(0)

if y_true.nunique() == 2:

    roc_auc = roc_auc_score(y_true, y_score)
    pr_auc = average_precision_score(y_true, y_score)

    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"PR-AUC:  {pr_auc:.4f}")

else:
    print("AUC cannot be calculated because only one class is present.")

ROC-AUC: 0.5218
PR-AUC:  0.1771


In [77]:
# ============================================================
# 5. BEST THRESHOLD FOR GOLD EVIDENCE
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.05, 0.96, 0.05)

results = []

for threshold in thresholds:

    pred = (y_score >= threshold).astype(int)

    precision = precision_score(
        y_true, pred, zero_division=0
    )

    recall = recall_score(
        y_true, pred, zero_division=0
    )

    f1 = f1_score(
        y_true, pred, zero_division=0
    )

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "TP": int(((pred == 1) & (y_true == 1)).sum()),
        "FP": int(((pred == 1) & (y_true == 0)).sum()),
        "FN": int(((pred == 0) & (y_true == 1)).sum())
    })

threshold_df = pd.DataFrame(results)

display(
    threshold_df.sort_values(
        "f1",
        ascending=False
    ).head(10)
)

best_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

print("\nBEST THRESHOLD:")
print(best_row)

,threshold,precision,recall,f1,TP,FP,FN
0,0.05,0.190866,0.698041,0.299766,1283,5439,555
1,0.10,0.180969,0.390098,0.247241,717,3245,1121
2,0.15,0.168389,0.239391,0.197708,440,2173,1398
3,0.20,0.151284,0.118607,0.132967,218,1223,1620
4,0.25,0.138968,0.052775,0.076498,97,601,1741
5,0.30,0.118881,0.027748,0.044993,51,378,1787
6,0.35,0.106667,0.017410,0.029935,32,268,1806
7,0.40,0.112745,0.012514,0.022527,23,181,1815
8,0.45,0.129771,0.009249,0.017268,17,114,1821
9,0.50,0.142857,0.007617,0.014463,14,84,1824



BEST THRESHOLD:
threshold       0.050000
precision       0.190866
recall          0.698041
f1              0.299766
TP           1283.000000
FP           5439.000000
FN            555.000000
Name: 0, dtype: float64


In [78]:
# ============================================================
# 6. FINAL NOVELTY EVIDENCE RECALL@K
# ============================================================

df_ranked = df.sort_values(
    ["document_id", "label_id", "novelty_evidence_score"],
    ascending=[True, True, False]
).copy()


def recall_at_k(data, k):

    hits = 0
    total = 0

    for (doc, label), group in data.groupby(
        ["document_id", "label_id"]
    ):

        gold = set()

        for x in group["gold_span_set"]:
            gold.update(x)

        if len(gold) == 0:
            continue

        top_k = group.head(k)

        retrieved = (
            set(top_k["span_a"]) |
            set(top_k["span_b"])
        )

        if len(gold & retrieved) > 0:
            hits += 1

        total += 1

    return hits / total if total else 0


for k in [1, 3, 5, 10]:

    score = recall_at_k(df_ranked, k)

    print(
        f"Novelty Evidence Recall@{k}: "
        f"{score:.4f} ({score*100:.2f}%)"
    )

Novelty Evidence Recall@1: 0.2736 (27.36%)
Novelty Evidence Recall@3: 0.5228 (52.28%)
Novelty Evidence Recall@5: 0.6173 (61.73%)
Novelty Evidence Recall@10: 0.6580 (65.80%)


In [79]:
# ============================================================
# 7. GOLD RETRIEVABILITY / UPPER BOUND
# ============================================================

example_stats = []

for (doc, label), group in df.groupby(
    ["document_id", "label_id"]
):

    gold = set()

    for x in group["gold_span_set"]:
        gold.update(x)

    if len(gold) == 0:
        continue

    candidate_spans = (
        set(group["span_a"]) |
        set(group["span_b"])
    )

    hit = len(gold & candidate_spans) > 0

    example_stats.append({
        "document_id": doc,
        "label_id": label,
        "gold_count": len(gold),
        "candidate_span_count": len(candidate_spans),
        "retrievable": hit
    })

retrievability_df = pd.DataFrame(example_stats)

print(
    "Examples with gold evidence:",
    len(retrievability_df)
)

print(
    "Examples where gold is present in candidates:",
    retrievability_df["retrievable"].sum()
)

print(
    "Maximum possible evidence recall:",
    retrievability_df["retrievable"].mean()
)

Examples with gold evidence: 614
Examples where gold is present in candidates: 404
Maximum possible evidence recall: 0.6579804560260586


In [80]:
# ============================================================
# 8. TARGET DISTRIBUTION
# ============================================================

print("Target distribution:")
print(df["target"].value_counts(dropna=False))

print("\nTarget proportions:")
print(
    df["target"].value_counts(normalize=True)
)

Target distribution:
target
1    5190
0    4230
2     950
Name: count, dtype: int64

Target proportions:
target
1    0.500482
0    0.407907
2    0.091610
Name: proportion, dtype: float64


In [81]:
# ============================================================
# 9. CONFLICT-BASED TARGET PREDICTION
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

# Binary conflict interpretation:
# target == 0 -> non-conflict
# target > 0  -> conflict

y_pair_true = (df["target"] > 0).astype(int)

pair_score = df["strict_conflict_score"].fillna(0)

best_threshold = best_row["threshold"]

y_pair_pred = (
    pair_score >= best_threshold
).astype(int)

print("Classification Report:")
print(
    classification_report(
        y_pair_true,
        y_pair_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_pair_true,
        y_pair_pred
    )
)

Classification Report:
              precision    recall  f1-score   support

           0       0.44      0.38      0.41      4230
           1       0.61      0.67      0.64      6140

    accuracy                           0.55     10370
   macro avg       0.53      0.52      0.52     10370
weighted avg       0.54      0.55      0.54     10370


Confusion Matrix:
[[1614 2616]
 [2052 4088]]


In [82]:
# ============================================================
# 10. MULTICLASS TARGET CHECK
# ============================================================

print("Unique targets:")
print(sorted(df["target"].dropna().unique()))

print("\nCounts:")
print(
    df["target"].value_counts().sort_index()
)

Unique targets:
[np.int64(0), np.int64(1), np.int64(2)]

Counts:
target
0    4230
1    5190
2     950
Name: count, dtype: int64


In [83]:
# ============================================================
# 11. ML CLASSIFIER
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

feature_cols = [
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "aligned_conflict_score",
    "modality_conflict",
    "negation_conflict"
]

X = df[feature_cols].fillna(0)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Classification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.46      0.47       846
           1       0.56      0.50      0.53      1038
           2       0.10      0.18      0.13       190

    accuracy                           0.45      2074
   macro avg       0.38      0.38      0.38      2074
weighted avg       0.49      0.45      0.47      2074


Confusion Matrix:
[[386 325 135]
 [339 521 178]
 [ 65  90  35]]


In [84]:
# ============================================================
# 12. FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance_df)

,feature,importance
3,strict_conflict_score,0.280601
4,aligned_conflict_score,0.274404
0,core_prop_similarity,0.220382
1,proposition_jaccard,0.195802
2,scope_compatibility,0.018064
6,negation_conflict,0.005608
5,modality_conflict,0.005138


In [85]:
# ============================================================
# 13. ML CONFLICT PROBABILITY
# ============================================================

if hasattr(rf_model, "predict_proba"):

    proba = rf_model.predict_proba(X)

    class_to_index = {
        cls: i
        for i, cls in enumerate(rf_model.classes_)
    }

    # Probability of any conflict target > 0
    conflict_indices = [
        class_to_index[c]
        for c in rf_model.classes_
        if c > 0
    ]

    df["ml_conflict_probability"] = (
        proba[:, conflict_indices].sum(axis=1)
    )

else:

    df["ml_conflict_probability"] = (
        rf_model.predict(X) > 0
    ).astype(float)

print(
    df["ml_conflict_probability"].describe()
)

count    10370.000000
mean         0.600510
std          0.275354
min          0.000000
25%          0.379305
50%          0.652970
75%          0.837918
max          1.000000
Name: ml_conflict_probability, dtype: float64


In [86]:
# ============================================================
# 14. OLD SCORE VS ML SCORE
# ============================================================

comparison = df[
    [
        "document_id",
        "label_id",
        "span_a",
        "span_b",
        "target",
        "strict_conflict_score",
        "aligned_conflict_score",
        "novelty_evidence_score",
        "ml_conflict_probability",
        "pair_touches_gold",
        "text_a",
        "text_b"
    ]
].copy()

display(
    comparison.sort_values(
        "ml_conflict_probability",
        ascending=False
    ).head(30)
)

,document_id,label_id,span_a,span_b,target,strict_conflict_score,aligned_conflict_score,novelty_evidence_score,ml_conflict_probability,pair_touches_gold,text_a,text_b
4660,440,nda-17,18,21,2,0.297222,0.001559,0.098506,1.0,False,The party receiving the Confidential Informati...,disclosed to the Receiving Party;
4631,440,nda-13,18,21,1,0.297222,0.001559,0.098506,1.0,False,The party receiving the Confidential Informati...,disclosed to the Receiving Party;
4624,440,nda-12,18,21,1,0.297222,0.001559,0.098506,1.0,False,The party receiving the Confidential Informati...,disclosed to the Receiving Party;
4730,440,nda-5,21,18,1,0.297222,0.001559,0.098506,1.0,False,disclosed to the Receiving Party;,The party receiving the Confidential Informati...
4740,440,nda-7,18,21,1,0.297222,0.001559,0.098506,1.0,True,The party receiving the Confidential Informati...,disclosed to the Receiving Party;
4700,440,nda-20,18,21,2,0.297222,0.001559,0.098506,1.0,False,The party receiving the Confidential Informati...,disclosed to the Receiving Party;
6599,563,nda-4,65,18,1,0.258435,0.001658,0.133159,1.0,False,20. To the extent that any Confidential Inform...,(b) known by the party receiving the Confident...
1608,20,nda-17,28,21,1,0.500000,0.003317,0.215209,1.0,False,3. Return of Confidential Information,iv. is independently developed by the Receivin...
6622,563,nda-8,18,65,1,0.258435,0.001658,0.133159,1.0,False,(b) known by the party receiving the Confident...,20. To the extent that any Confidential Inform...
1603,20,nda-17,29,21,1,0.356250,0.004256,0.196387,1.0,False,3.1 All Confidential Information disclosed by ...,iv. is independently developed by the Receivin...


In [87]:
# ============================================================
# 15. ML RANKING RECALL@K
# ============================================================

ml_ranked = df.sort_values(
    ["document_id", "label_id", "ml_conflict_probability"],
    ascending=[True, True, False]
).copy()


for k in [1, 3, 5, 10]:

    score = recall_at_k(
        ml_ranked,
        k
    )

    print(
        f"ML Evidence Recall@{k}: "
        f"{score:.4f} ({score*100:.2f}%)"
    )

ML Evidence Recall@1: 0.3371 (33.71%)
ML Evidence Recall@3: 0.5765 (57.65%)
ML Evidence Recall@5: 0.6531 (65.31%)
ML Evidence Recall@10: 0.6580 (65.80%)


In [88]:
# ============================================================
# 16. COMPARE RANKING METHODS
# ============================================================

methods = {
    "Novelty Evidence": "novelty_evidence_score",
    "Strict Conflict": "strict_conflict_score",
    "Aligned Conflict": "aligned_conflict_score",
    "ML Conflict": "ml_conflict_probability"
}

comparison_results = []

for method_name, score_col in methods.items():

    ranked = df.sort_values(
        ["document_id", "label_id", score_col],
        ascending=[True, True, False]
    )

    row = {
        "method": method_name
    }

    for k in [1, 3, 5, 10]:
        row[f"Recall@{k}"] = recall_at_k(
            ranked,
            k
        )

    comparison_results.append(row)

comparison_results_df = pd.DataFrame(
    comparison_results
)

display(comparison_results_df)

,method,Recall@1,Recall@3,Recall@5,Recall@10
0,Novelty Evidence,0.273616,0.522801,0.617264,0.65798
1,Strict Conflict,0.293160,0.530945,0.627036,0.65798
2,Aligned Conflict,0.294788,0.511401,0.612378,0.65798
3,ML Conflict,0.337134,0.576547,0.653094,0.65798


In [89]:
# ============================================================
# 17. BEST METHOD
# ============================================================

best_method = comparison_results_df.loc[
    comparison_results_df["Recall@10"].idxmax()
]

print("Best method by Recall@10:")
print(best_method)

Best method by Recall@10:
method       Novelty Evidence
Recall@1             0.273616
Recall@3             0.522801
Recall@5             0.617264
Recall@10             0.65798
Name: 0, dtype: object


In [90]:
# ============================================================
# 18. SAVE CURRENT PROJECT STATE
# ============================================================

import os

save_dir = "/kaggle/working/project_outputs"

os.makedirs(
    save_dir,
    exist_ok=True
)

df.to_pickle(
    f"{save_dir}/dev_pairs_with_scores.pkl"
)

comparison_results_df.to_csv(
    f"{save_dir}/ranking_method_comparison.csv",
    index=False
)

importance_df.to_csv(
    f"{save_dir}/feature_importance.csv",
    index=False
)

retrievability_df.to_csv(
    f"{save_dir}/gold_retrievability.csv",
    index=False
)

print("Saved files:")
print(os.listdir(save_dir))

Saved files:
['feature_importance.csv', 'dev_pairs_with_scores.pkl', 'ranking_method_comparison.csv', 'gold_retrievability.csv']


In [94]:
# ============================================================
# RECREATE MISSING MODEL FEATURES
# ============================================================

# Make a copy so the original dataframe is preserved
dev_pair_df = dev_pair_df.copy()

# ------------------------------------------------------------
# 1. Number of candidate pairs for each document + label
# ------------------------------------------------------------

dev_pair_df["pair_count"] = (
    dev_pair_df
    .groupby(["document_id", "label_id"])["span_a"]
    .transform("size")
)


# ------------------------------------------------------------
# 2. Number of structured-conflict candidate pairs
# ------------------------------------------------------------

dev_pair_df["structured_candidate_count"] = (
    dev_pair_df
    .groupby(["document_id", "label_id"])["structured_conflict_candidate"]
    .transform(
        lambda x: x.fillna(False).astype(bool).sum()
    )
)


# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print("Recreated model features:")

print(
    dev_pair_df[
        [
            "pair_count",
            "structured_candidate_count"
        ]
    ].describe()
)

print("\nColumns now available:")
print(
    "pair_count:",
    "pair_count" in dev_pair_df.columns
)

print(
    "structured_candidate_count:",
    "structured_candidate_count" in dev_pair_df.columns
)


# ------------------------------------------------------------
# CHECK FOR ANY OTHER MISSING FEATURES
# ------------------------------------------------------------

if "model_feature_names" in globals():

    missing = [
        f for f in model_feature_names
        if f not in dev_pair_df.columns
    ]

    print("\nMissing model features:", missing)

elif "feature_names" in globals():

    missing = [
        f for f in feature_names
        if f not in dev_pair_df.columns
    ]

    print("\nMissing model features:", missing)

else:
    print("\nThe two missing features have been recreated.")

Recreated model features:
       pair_count  structured_candidate_count
count     10370.0                10370.000000
mean         10.0                    0.170685
std           0.0                    0.533196
min          10.0                    0.000000
25%          10.0                    0.000000
50%          10.0                    0.000000
75%          10.0                    0.000000
max          10.0                    4.000000

Columns now available:
pair_count: True
structured_candidate_count: True

The two missing features have been recreated.


In [95]:
# ============================================================
# FIX: RECREATE FEATURES EXACTLY AS EXPECTED BY THE FITTED MODEL
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Get the exact feature names used when the model was fitted
# ------------------------------------------------------------

expected_features = list(model.feature_names_in_)

print("Model expects:", len(expected_features), "features")


# ------------------------------------------------------------
# 2. Recreate missing aggregated features
#    Aggregation is done within each document + label
# ------------------------------------------------------------

group_cols = ["document_id", "label_id"]

# Base numeric features available in the current dataframe
base_features = [
    "core_prop_similarity",
    "proposition_jaccard",
    "scope_compatibility",
    "strict_conflict_score",
    "aligned_conflict_score",
    "modality_conflict",
    "negation_conflict",
    "clause_reasoning_score_a",
    "clause_reasoning_score_b",
    "proposition_similarity",
    "condition_similarity",
    "party_similarity",
    "structured_conflict_score",
    "proposition_overlap",
    "aligned_conflict",
    "scope_aware_score",
]

# Only use features that actually exist
base_features = [
    f for f in base_features
    if f in dev_pair_df.columns
]


# ------------------------------------------------------------
# 3. Create common aggregate features
# ------------------------------------------------------------

for f in base_features:

    # Mean
    mean_col = f"{f}_mean"
    if mean_col in expected_features and mean_col not in dev_pair_df.columns:
        dev_pair_df[mean_col] = (
            dev_pair_df.groupby(group_cols)[f]
            .transform("mean")
        )

    # Maximum
    max_col = f"{f}_max"
    if max_col in expected_features and max_col not in dev_pair_df.columns:
        dev_pair_df[max_col] = (
            dev_pair_df.groupby(group_cols)[f]
            .transform("max")
        )

    # Minimum
    min_col = f"{f}_min"
    if min_col in expected_features and min_col not in dev_pair_df.columns:
        dev_pair_df[min_col] = (
            dev_pair_df.groupby(group_cols)[f]
            .transform("min")
        )

    # Count
    count_col = f"{f}_count"
    if count_col in expected_features and count_col not in dev_pair_df.columns:
        dev_pair_df[count_col] = (
            dev_pair_df.groupby(group_cols)[f]
            .transform("count")
        )


# ------------------------------------------------------------
# 4. Handle special conflict count features
# ------------------------------------------------------------

for f in [
    "aligned_conflict",
    "modality_conflict",
    "negation_conflict",
    "structured_conflict_candidate",
]:

    if f in dev_pair_df.columns:

        count_col = f"{f}_count"

        if count_col in expected_features and count_col not in dev_pair_df.columns:

            dev_pair_df[count_col] = (
                dev_pair_df.groupby(group_cols)[f]
                .transform(
                    lambda x: pd.to_numeric(x, errors="coerce").fillna(0).sum()
                )
            )


# ------------------------------------------------------------
# 5. Recreate any expected feature from available raw feature
# ------------------------------------------------------------

for expected in expected_features:

    if expected in dev_pair_df.columns:
        continue

    # If model expects a *_mean / *_max / *_min / *_count feature
    # but the base feature exists, create it.
    for suffix, operation in [
        ("_mean", "mean"),
        ("_max", "max"),
        ("_min", "min"),
        ("_count", "count"),
    ]:

        if expected.endswith(suffix):

            base = expected[:-len(suffix)]

            if base in dev_pair_df.columns:

                if operation == "mean":
                    dev_pair_df[expected] = (
                        dev_pair_df.groupby(group_cols)[base]
                        .transform("mean")
                    )

                elif operation == "max":
                    dev_pair_df[expected] = (
                        dev_pair_df.groupby(group_cols)[base]
                        .transform("max")
                    )

                elif operation == "min":
                    dev_pair_df[expected] = (
                        dev_pair_df.groupby(group_cols)[base]
                        .transform("min")
                    )

                elif operation == "count":
                    dev_pair_df[expected] = (
                        dev_pair_df.groupby(group_cols)[base]
                        .transform("count")
                    )


# ------------------------------------------------------------
# 6. Check for anything still missing
# ------------------------------------------------------------

missing = [
    f for f in expected_features
    if f not in dev_pair_df.columns
]

print("\nStill missing:", missing)


# ------------------------------------------------------------
# 7. If nothing is missing, create EXACT model input
# ------------------------------------------------------------

if len(missing) > 0:

    raise ValueError(
        "These model features could not be recreated: "
        + str(missing)
    )

X_ml = dev_pair_df[expected_features].copy()


# ------------------------------------------------------------
# 8. Clean numeric values
# ------------------------------------------------------------

X_ml = X_ml.apply(pd.to_numeric, errors="coerce")
X_ml = X_ml.replace([np.inf, -np.inf], np.nan)
X_ml = X_ml.fillna(0)


# ------------------------------------------------------------
# 9. Predict using the already-fitted model
# ------------------------------------------------------------

dev_pair_df["ml_conflict_probability"] = model.predict_proba(X_ml)[:, 1]


# ------------------------------------------------------------
# 10. Verify
# ------------------------------------------------------------

print("\nML probability:")
print(dev_pair_df["ml_conflict_probability"].describe())

print("\nML feature matrix shape:", X_ml.shape)
print("Expected feature count:", len(expected_features))

Model expects: 30 features

Still missing: []

ML probability:
count    10370.000000
mean         0.260150
std          0.137975
min          0.009621
25%          0.156252
50%          0.238118
75%          0.330216
max          0.715046
Name: ml_conflict_probability, dtype: float64

ML feature matrix shape: (10370, 30)
Expected feature count: 30


In [98]:
# ============================================================
# FIX: RECREATE pair_touches_gold
# ============================================================

import ast
import pandas as pd

# ------------------------------------------------------------
# 1. Parse gold_spans safely
# ------------------------------------------------------------

def parse_span_set(x):
    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if x is None:
        return set()

    if isinstance(x, str):
        try:
            value = ast.literal_eval(x)
            if isinstance(value, (list, tuple, set)):
                return set(value)
        except:
            pass

    return set()


# ------------------------------------------------------------
# 2. Create gold_span_set
# ------------------------------------------------------------

dev_pair_df["gold_span_set"] = (
    dev_pair_df["gold_spans"].apply(parse_span_set)
)


# ------------------------------------------------------------
# 3. Check whether either span touches gold
# ------------------------------------------------------------

dev_pair_df["pair_touches_gold"] = dev_pair_df.apply(
    lambda r:
        (r["span_a"] in r["gold_span_set"]) or
        (r["span_b"] in r["gold_span_set"]),
    axis=1
)


# ------------------------------------------------------------
# 4. Verify
# ------------------------------------------------------------

print("pair_touches_gold created successfully.")

print(
    dev_pair_df["pair_touches_gold"]
    .value_counts()
)

print("\nGold-touching pairs:")
print(
    dev_pair_df["pair_touches_gold"].sum()
)

pair_touches_gold created successfully.
pair_touches_gold
False    8532
True     1838
Name: count, dtype: int64

Gold-touching pairs:
1838


In [99]:
# ============================================================
# FINAL HYBRID EVIDENCE SCORE
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Normalize a series safely
# ------------------------------------------------------------

def minmax_normalize(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)

    mn = s.min()
    mx = s.max()

    if mx == mn:
        return pd.Series(0.0, index=s.index)

    return (s - mn) / (mx - mn)


# ------------------------------------------------------------
# 2. Normalize available evidence/conflict signals
# ------------------------------------------------------------

dev_pair_df["ml_conflict_norm"] = minmax_normalize(
    dev_pair_df["ml_conflict_probability"]
)

dev_pair_df["strict_conflict_norm"] = minmax_normalize(
    dev_pair_df["strict_conflict_score"]
)

dev_pair_df["aligned_conflict_norm"] = minmax_normalize(
    dev_pair_df["aligned_conflict_score"]
)

dev_pair_df["core_similarity_norm"] = minmax_normalize(
    dev_pair_df["core_prop_similarity"]
)

dev_pair_df["proposition_jaccard_norm"] = minmax_normalize(
    dev_pair_df["proposition_jaccard"]
)


# ------------------------------------------------------------
# 3. Final hybrid score
# ------------------------------------------------------------

dev_pair_df["hybrid_evidence_score"] = (
    0.40 * dev_pair_df["ml_conflict_norm"]
    + 0.25 * dev_pair_df["strict_conflict_norm"]
    + 0.15 * dev_pair_df["aligned_conflict_norm"]
    + 0.10 * dev_pair_df["core_similarity_norm"]
    + 0.10 * dev_pair_df["proposition_jaccard_norm"]
)


# ------------------------------------------------------------
# 4. Check result
# ------------------------------------------------------------

print("Hybrid Evidence Score:")
print(dev_pair_df["hybrid_evidence_score"].describe())


# ------------------------------------------------------------
# 5. Show top-ranked pairs
# ------------------------------------------------------------

display(
    dev_pair_df[
        [
            "document_id",
            "label_id",
            "span_a",
            "span_b",
            "target",
            "strict_conflict_score",
            "aligned_conflict_score",
            "ml_conflict_probability",
            "hybrid_evidence_score",
            "pair_touches_gold"
        ]
    ]
    .sort_values("hybrid_evidence_score", ascending=False)
    .head(20)
)

Hybrid Evidence Score:
count    10370.000000
mean         0.216949
std          0.125167
min          0.000000
25%          0.123111
50%          0.188919
75%          0.295109
max          0.887069
Name: hybrid_evidence_score, dtype: float64


,document_id,label_id,span_a,span_b,target,strict_conflict_score,aligned_conflict_score,ml_conflict_probability,hybrid_evidence_score,pair_touches_gold
7240,590,nda-2,7,11,2,0.900000,0.762500,0.559974,0.887069,False
7260,590,nda-3,11,7,1,0.900000,0.762500,0.416449,0.805686,False
7147,590,nda-1,11,7,0,0.900000,0.762500,0.372554,0.780795,False
9467,73,nda-20,133,37,2,0.850000,0.100000,0.529595,0.750626,False
9420,73,nda-17,133,37,2,0.850000,0.100000,0.513852,0.741699,True
9791,79,nda-2,10,5,2,0.730000,0.070000,0.663949,0.720908,False
9361,73,nda-10,133,37,1,0.850000,0.100000,0.466945,0.715101,True
9490,73,nda-5,133,37,1,0.850000,0.100000,0.464502,0.713716,True
9485,73,nda-4,133,37,1,0.850000,0.100000,0.460614,0.711512,True
9350,73,nda-1,185,137,2,0.783333,0.091667,0.582026,0.710198,False


In [100]:
# ============================================================
# HYBRID EVIDENCE RECALL@K
# ============================================================

def recall_at_k_hybrid(data, k):
    hits = 0
    total = 0

    for (doc, label), group in data.groupby(
        ["document_id", "label_id"]
    ):

        gold = set()

        for x in group["gold_span_set"]:
            gold.update(x)

        if len(gold) == 0:
            continue

        group = group.sort_values(
            "hybrid_evidence_score",
            ascending=False
        )

        top_k = group.head(k)

        retrieved = (
            set(top_k["span_a"]) |
            set(top_k["span_b"])
        )

        if len(gold & retrieved) > 0:
            hits += 1

        total += 1

    return hits / total if total else 0


hybrid_results = {}

for k in [1, 3, 5, 10]:
    score = recall_at_k_hybrid(dev_pair_df, k)

    hybrid_results[k] = score

    print(
        f"Hybrid Evidence Recall@{k}: "
        f"{score:.4f} ({score*100:.2f}%)"
    )

Hybrid Evidence Recall@1: 0.2899 (28.99%)
Hybrid Evidence Recall@3: 0.5244 (52.44%)
Hybrid Evidence Recall@5: 0.6270 (62.70%)
Hybrid Evidence Recall@10: 0.6580 (65.80%)


In [101]:
# ============================================================
# COMPARE ALL EVIDENCE RANKING METHODS
# ============================================================

comparison = pd.DataFrame({
    "Method": [
        "Novelty Evidence",
        "Strict Conflict",
        "Aligned Conflict",
        "ML Conflict",
        "Hybrid Evidence"
    ],

    "Recall@1": [
        0.273616,
        0.293160,
        0.294788,
        0.337134,
        hybrid_results[1]
    ],

    "Recall@3": [
        0.522801,
        0.530945,
        0.511401,
        0.576547,
        hybrid_results[3]
    ],

    "Recall@5": [
        0.617264,
        0.627036,
        0.612378,
        0.653094,
        hybrid_results[5]
    ],

    "Recall@10": [
        0.657980,
        0.657980,
        0.657980,
        0.657980,
        hybrid_results[10]
    ]
})

display(comparison)

print("\nBest method by Recall@1:")
print(comparison.loc[comparison["Recall@1"].idxmax()])

print("\nBest method by Recall@3:")
print(comparison.loc[comparison["Recall@3"].idxmax()])

print("\nBest method by Recall@5:")
print(comparison.loc[comparison["Recall@5"].idxmax()])

print("\nBest method by Recall@10:")
print(comparison.loc[comparison["Recall@10"].idxmax()])

,Method,Recall@1,Recall@3,Recall@5,Recall@10
0,Novelty Evidence,0.273616,0.522801,0.617264,0.65798
1,Strict Conflict,0.293160,0.530945,0.627036,0.65798
2,Aligned Conflict,0.294788,0.511401,0.612378,0.65798
3,ML Conflict,0.337134,0.576547,0.653094,0.65798
4,Hybrid Evidence,0.289902,0.524430,0.627036,0.65798



Best method by Recall@1:
Method       ML Conflict
Recall@1        0.337134
Recall@3        0.576547
Recall@5        0.653094
Recall@10        0.65798
Name: 3, dtype: object

Best method by Recall@3:
Method       ML Conflict
Recall@1        0.337134
Recall@3        0.576547
Recall@5        0.653094
Recall@10        0.65798
Name: 3, dtype: object

Best method by Recall@5:
Method       ML Conflict
Recall@1        0.337134
Recall@3        0.576547
Recall@5        0.653094
Recall@10        0.65798
Name: 3, dtype: object

Best method by Recall@10:
Method       Hybrid Evidence
Recall@1            0.289902
Recall@3             0.52443
Recall@5            0.627036
Recall@10            0.65798
Name: 4, dtype: object


In [102]:
# ============================================================
# FINAL ML MODEL EVALUATION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Verify required columns
# ------------------------------------------------------------

required_cols = [
    "target",
    "ml_conflict_probability",
    "pair_touches_gold"
]

missing = [c for c in required_cols if c not in dev_pair_df.columns]

print("Missing columns:", missing)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# 2. Binary target
# ------------------------------------------------------------

y_true = (dev_pair_df["target"] > 0).astype(int)

y_prob = pd.to_numeric(
    dev_pair_df["ml_conflict_probability"],
    errors="coerce"
).fillna(0)

# ------------------------------------------------------------
# 3. ROC-AUC and PR-AUC
# ------------------------------------------------------------

roc_auc = roc_auc_score(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)

print("=" * 60)
print("ML CONFLICT MODEL EVALUATION")
print("=" * 60)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")

# ------------------------------------------------------------
# 4. Find best F1 threshold
# ------------------------------------------------------------

thresholds = np.arange(0.05, 0.96, 0.05)

results = []

for threshold in thresholds:

    y_pred = (y_prob >= threshold).astype(int)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "TP": int(((y_true == 1) & (y_pred == 1)).sum()),
        "FP": int(((y_true == 0) & (y_pred == 1)).sum()),
        "FN": int(((y_true == 1) & (y_pred == 0)).sum())
    })

threshold_df = pd.DataFrame(results)

best_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

print("\nBest threshold:")
print(best_row)

# ------------------------------------------------------------
# 5. Classification report at best threshold
# ------------------------------------------------------------

best_threshold = best_row["threshold"]

y_best = (
    y_prob >= best_threshold
).astype(int)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_best,
        target_names=[
            "No Conflict",
            "Conflict"
        ],
        zero_division=0
    )
)

# ------------------------------------------------------------
# 6. Confusion matrix
# ------------------------------------------------------------

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_best))

# ------------------------------------------------------------
# 7. Threshold comparison
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("THRESHOLD PERFORMANCE")
print("=" * 60)

display(
    threshold_df.round(4)
)

# ------------------------------------------------------------
# 8. Gold evidence statistics
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GOLD EVIDENCE STATISTICS")
print("=" * 60)

print(
    "Gold-touching pairs:",
    dev_pair_df["pair_touches_gold"].value_counts()
)

# ------------------------------------------------------------
# 9. ML probability by gold contact
# ------------------------------------------------------------

gold_stats = (
    dev_pair_df
    .groupby("pair_touches_gold")["ml_conflict_probability"]
    .agg(["count", "mean", "median", "max"])
)

print("\nML probability by gold contact:")
display(gold_stats)

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL ML SUMMARY")
print("=" * 60)

print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")
print(f"Best F1 : {best_row['f1']:.4f}")
print(f"Threshold: {best_threshold:.2f}")
print(f"Precision: {best_row['precision']:.4f}")
print(f"Recall   : {best_row['recall']:.4f}")

Missing columns: []
ML CONFLICT MODEL EVALUATION
ROC-AUC: 0.6107
PR-AUC:  0.7332

Best threshold:
threshold       0.050000
precision       0.601385
recall          0.990228
f1              0.748308
TP           6080.000000
FP           4030.000000
FN             60.000000
Name: 0, dtype: float64

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 No Conflict       0.77      0.05      0.09      4230
    Conflict       0.60      0.99      0.75      6140

    accuracy                           0.61     10370
   macro avg       0.69      0.52      0.42     10370
weighted avg       0.67      0.61      0.48     10370

Confusion Matrix:
[[ 200 4030]
 [  60 6080]]

THRESHOLD PERFORMANCE


,threshold,precision,recall,f1,TP,FP,FN
0,0.05,0.6014,0.9902,0.7483,6080,4030,60
1,0.10,0.6108,0.9381,0.7399,5760,3670,380
2,0.15,0.6263,0.8078,0.7055,4960,2960,1180
3,0.20,0.6260,0.6678,0.6462,4100,2450,2040
4,0.25,0.6559,0.4967,0.5653,3050,1600,3090
5,0.30,0.7226,0.3860,0.5032,2370,910,3770
6,0.35,0.8036,0.2932,0.4296,1800,440,4340
7,0.40,0.8439,0.2378,0.3710,1460,270,4680
8,0.45,0.9187,0.1840,0.3066,1130,100,5010
9,0.50,0.9762,0.1336,0.2350,820,20,5320



GOLD EVIDENCE STATISTICS
Gold-touching pairs: pair_touches_gold
False    8532
True     1838
Name: count, dtype: int64

ML probability by gold contact:


,count,mean,median,max
pair_touches_gold,,,,
False,8532,0.258441,0.237835,0.715046
True,1838,0.268086,0.239759,0.715046



FINAL ML SUMMARY
ROC-AUC : 0.6107
PR-AUC  : 0.7332
Best F1 : 0.7483
Threshold: 0.05
Precision: 0.6014
Recall   : 0.9902
